# Paper 4 — From Survey Archive to Decision Engine
**An AI-Augmented Pipeline for Reactivating Legacy Stakeholder Data in Urban Living Labs**

Truong Tho Living Lab (TTLL), Thu Duc City, Ho Chi Minh City, Vietnam.
Target journal: *Sustainable Cities and Society* (Elsevier, Q1).

This notebook runs the complete analysis end to end from the raw survey workbooks
and the open-data feature table verified in Paper 1. Every number reported in the
manuscript is produced here and written to `Results/`, `Tables/` and `Figures/`.

**Pipeline layers implemented here**

| Layer | Name | Notebook step |
|---|---|---|
| L1 | Harmonisation | Step 1 |
| L2 | Quality assurance | Step 2 |
| L3 | Enrichment | Step 3 |
| L4 | Inference (ML + SHAP + LLM coding) | Steps 4–8 |
| L5 | Decision interface | Step 10 |


## Step 0 — Configuration and environment


In [1]:
import os, sys, json, math, re, warnings, itertools, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
RNG = 20260906
np.random.seed(RNG)

# ---------------------------------------------------------------- paths
HERE = Path.cwd()
# The notebook lives in <project>/Paper4/code-p4/. Walk up until we find Data/.
ROOT = HERE
for _ in range(4):
    if (ROOT / "Data").is_dir() and (ROOT / "Paper1").is_dir():
        break
    ROOT = ROOT.parent
if not (ROOT / "Data").is_dir():
    raise SystemExit("Could not locate project root containing Data/ and Paper1/")

P4       = ROOT / "Paper4"
DATA     = ROOT / "Data"
P1RES    = ROOT / "Paper1" / "Results"
P3DER    = ROOT / "Paper3" / "Derived"
DATA4    = P4 / "DataPaper4"

FIG      = P4 / "Figures";   FIG.mkdir(parents=True, exist_ok=True)
TAB      = P4 / "Tables";    TAB.mkdir(parents=True, exist_ok=True)
RES      = P4 / "Results";   RES.mkdir(parents=True, exist_ok=True)
COD      = P4 / "Coding";    COD.mkdir(parents=True, exist_ok=True)
DASH     = P4 / "code-p4" / "Dashboard"; DASH.mkdir(parents=True, exist_ok=True)

print("project root :", ROOT)
print("paper folder :", P4)

# ---------------------------------------------------------------- switches
RUN_SHAP        = True    # SHAP is fast at this n; keep on
RUN_NESTED_CV   = True    # 5x5 nested CV, ~1-3 min
N_PERM          = 5000    # permutation iterations for the barrier tests
N_BOOT          = 2000    # bootstrap iterations for performance CIs

RESULTS = {}   # every reported scalar lands here and is written to Results/paper4_results.json


def rec(key, value):
    """Record a result value and echo it."""
    RESULTS[key] = value
    return value


def save_table(df, name, caption=""):
    """Write a table to Tables/ as CSV and register it."""
    path = TAB / f"{name}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig")
    RESULTS.setdefault("_tables", {})[name] = {"rows": int(len(df)), "caption": caption}
    print(f"  -> Tables/{name}.csv  ({len(df)} rows)")
    return df


project root : /home/claude/devroot
paper folder : /home/claude/devroot/Paper4


### Step 0.1 — Open-data completeness audit (Layer 2, first gate)

Paper 4 reuses the open-data feature table verified in Paper 1 rather than the
`DataPaper4/` export. This cell documents *why*: it measures how many of the 539
geolocated survey points fall inside each raster in `DataPaper4/`, and compares
that with the coverage of the Paper 1 feature table.

This audit is not housekeeping — it is Layer 2 of the pipeline running on itself,
and the reason it exists is Barrier #3 in the expert diagnosis (*data access*, 61.0%).


In [2]:
audit_rows = []
try:
    import rasterio
    HAVE_RASTERIO = True
except Exception:
    HAVE_RASTERIO = False
    print("rasterio not installed - skipping the DataPaper4 raster audit "
          "(the audit result is cached in Results/ and does not affect the analysis)")

_mt_probe = pd.read_csv(P1RES / "master_table.csv")
_pts = _mt_probe[["lon", "lat"]].dropna().values

if HAVE_RASTERIO and DATA4.is_dir():
    for tif in sorted(DATA4.glob("*/*.tif")):
        with rasterio.open(tif) as src:
            epsg = src.crs.to_epsg() if src.crs else None
            if epsg == 4326:
                b = src.bounds
                inside = ((_pts[:, 0] >= b.left) & (_pts[:, 0] <= b.right) &
                          (_pts[:, 1] >= b.bottom) & (_pts[:, 1] <= b.top))
                vals = np.array([v[0] for v in src.sample(_pts)], dtype=float)
                nd = src.nodata
                ok = np.isfinite(vals) & ((vals != nd) if nd is not None else True)
                pct_in, pct_ok = 100 * inside.mean(), 100 * ok.mean()
            else:
                pct_in = pct_ok = np.nan
            audit_rows.append({
                "layer": tif.parent.name, "file": tif.name, "crs": str(src.crs),
                "width": src.width, "height": src.height, "bands": src.count,
                "pct_points_in_bbox": round(pct_in, 1) if pct_in == pct_in else None,
                "pct_points_valid":   round(pct_ok, 1) if pct_ok == pct_ok else None,
            })

audit = pd.DataFrame(audit_rows)
if len(audit):
    print(audit.to_string(index=False))
    worst = audit["pct_points_valid"].min()
    print(f"\nDataPaper4 worst-case point coverage: {worst}%")
    rec("datapaper4_worst_coverage_pct", float(worst) if worst == worst else None)
    save_table(audit, "S1_datapaper4_raster_audit",
               "Coverage audit of the DataPaper4 export against the 539 survey points.")

# ---- coverage of the Paper 1 feature table (what this paper actually uses)
OPEN_FEATURES = [
    "elev_m", "slope_deg", "elev_m_r300", "hand_m",
    "flood_freq", "water_recur", "water_max", "flood_freq_r300",
    "flood_sar", "vh_p1", "vh_p2", "d_vh", "flood_sar_r300",
    "ndvi", "ndbi", "ndwi", "mndwi", "ndvi_r300", "ndbi_r300",
    "ndvi_p2", "ndbi_p2", "d_ndvi", "d_ndbi", "d_ndvi_r300",
    "lst_c", "lst_p2", "d_lst",
    "pop_dens", "pop_2025", "built_m2", "built_2025", "d_pop", "d_built", "built_frac",
    "d_water_m", "road_density", "d_metro_m",
]
cov = (1 - _mt_probe[OPEN_FEATURES].isna().mean()) * 100
print(f"\nPaper 1 feature table: {len(_mt_probe)} points x {len(OPEN_FEATURES)} open-data features")
print(f"  minimum per-feature coverage : {cov.min():.1f}%  ({cov.idxmin()})")
print(f"  features at 100% coverage    : {(cov >= 99.999).sum()} / {len(OPEN_FEATURES)}")
rec("p1_feature_min_coverage_pct", round(float(cov.min()), 1))
rec("p1_features_full_coverage", int((cov >= 99.999).sum()))
rec("n_open_features", len(OPEN_FEATURES))
assert cov.min() > 95, "Open-data feature coverage below 95% - stop and re-check the source."
print("\n  ALL LAYERS PASS.")


               layer                       file        crs  width  height  bands  pct_points_in_bbox  pct_points_valid
   02_ghsl_pop_built   ghsl_built_2020_100m.tif ESRI:54009     31      32      1                 NaN               NaN
   02_ghsl_pop_built   ghsl_built_2025_100m.tif ESRI:54009     31      32      1                 NaN               NaN
   02_ghsl_pop_built ghsl_built_change_100m.tif ESRI:54009     31      32      1                 NaN               NaN
   02_ghsl_pop_built     ghsl_pop_2020_100m.tif ESRI:54009     31      32      1                 NaN               NaN
   02_ghsl_pop_built     ghsl_pop_2025_100m.tif ESRI:54009     31      32      1                 NaN               NaN
   02_ghsl_pop_built   ghsl_pop_change_100m.tif ESRI:54009     31      32      1                 NaN               NaN
 03_sentinel_indices         s1_sar_vh_2023.tif  EPSG:4326    279     279      1                32.8             100.0
 03_sentinel_indices        s2_indices_2023.tif 

## Step 1 — Layer 1: Harmonisation

Four instruments with incompatible schemas (Citizens n=460, SMEs n=80, Experts n=41,
Authorities n=23) are mapped onto one machine-readable schema. The harmonisation
codebook is written to `Results/codebook.json` and is a deliverable of the pipeline,
not a by-product.


In [3]:
CIT = pd.read_excel(DATA / "QData_Citizens.xlsx")
SME = pd.read_excel(DATA / "QData_SMEs.xlsx")
EXP = pd.read_excel(DATA / "QData_Experts.xlsx")
AUT = pd.read_excel(DATA / "QData_Authorities.xlsx")
print(f"Citizens {CIT.shape} | SMEs {SME.shape} | Experts {EXP.shape} | Authorities {AUT.shape}")
rec("n_citizens", len(CIT)); rec("n_smes", len(SME))
rec("n_experts", len(EXP)); rec("n_authorities", len(AUT))

# ---- canonical scales -------------------------------------------------------
# Six distinct Likert families appear across the four instruments. Layer 1 maps
# each to a common 1-5 severity/effectiveness direction.
SCALE_SEVERITY = {"Không có": 1, "Thấp": 2, "Trung bình": 3, "Cao": 4, "Nguy hiểm": 5}
SCALE_EFFECT   = {"Rất khó khăn": 1, "Còn nhiều khó khăn": 2, "Trung bình": 3,
                  "Hiệu quả": 4, "Rất hiệu quả": 5}
SCALE_AGREE    = {"Rất không đồng ý": 1, "Không đồng ý": 2, "Không ý kiến": 3,
                  "Đồng ý": 4, "Rất đồng ý": 5}

CODEBOOK = {
    "instruments": {
        "Citizens":    {"n": len(CIT), "role": "ML training set (RQ3)"},
        "SMEs":        {"n": len(SME), "role": "transfer validation set (RQ3)"},
        "Experts":     {"n": len(EXP), "role": "barrier diagnosis (RQ1)"},
        "Authorities": {"n": len(AUT), "role": "institutional comparison (RQ1)"},
    },
    "scales": {"severity_1_5": SCALE_SEVERITY,
               "effectiveness_1_5": SCALE_EFFECT,
               "agreement_1_5": SCALE_AGREE},
}

# ---- the six urban issues, harmonised across instruments --------------------
ISSUES = ["air", "flood", "noise", "water", "congest", "accident"]
ISSUE_EN = {"air": "Air pollution", "flood": "Flooding", "noise": "Noise pollution",
            "water": "Water pollution", "congest": "Traffic congestion",
            "accident": "Traffic accidents"}
# Expert / Authority section-C item numbers per issue
EXP_ISSUE_Q = {"air": "Q3.1", "flood": "Q3.2", "noise": "Q3.3",
               "water": "Q3.4", "congest": "Q3.5", "accident": "Q3.6"}

# ---- the five barrier classes of the Q3.xA matrix ---------------------------
BARRIER5 = {
    "A1": ("Technical / expertise / natural conditions", "TECHNICAL"),
    "A2": ("Financial",                                   "RESOURCE"),
    "A3": ("High-quality human resources",                "RESOURCE"),
    "A4": ("Policy and legal framework",                  "INSTITUTIONAL"),
    "A5": ("Social characteristics (awareness, culture)",  "SOCIAL"),
}
CODEBOOK["barrier_matrix"] = {k: {"label_en": v[0], "class": v[1]} for k, v in BARRIER5.items()}

# ---- Q2.10 digital-transformation barriers ---------------------------------
DT_BARRIERS = {
    "Cơ chế phối hợp giữa các bên":          ("Inter-agency coordination mechanisms", "INSTITUTIONAL"),
    "Chính sách chưa rõ ràng":               ("Unclear policy",                       "INSTITUTIONAL"),
    "Tiếp cận dữ liệu":                      ("Data access",                          "INSTITUTIONAL"),
    "Thiếu ngân sách":                       ("Insufficient budget",                  "RESOURCE"),
    "Chuyên môn khác nhau giữa ban ngành":   ("Divergent expertise across departments", "INSTITUTIONAL"),
    "Thiếu người thực hiện":                 ("Insufficient personnel",               "RESOURCE"),
    "Không cần chuyển đổi":                  ("No transformation needed",             "OTHER"),
}
CODEBOOK["dt_barriers"] = {v[0]: v[1] for v in DT_BARRIERS.values()}

with open(RES / "codebook.json", "w", encoding="utf-8") as fh:
    json.dump(CODEBOOK, fh, ensure_ascii=False, indent=2)
print("  -> Results/codebook.json")


Citizens (460, 54) | SMEs (80, 50) | Experts (41, 87) | Authorities (23, 61)
  -> Results/codebook.json


## Step 2 — Layer 2: Quality assurance

Automated detection of out-of-scale values, impossible coordinates and anomalous
missingness, run *before* any analysis. In this pipeline Layer 2 also re-derives the
reliability and spatial-structure statistics after enrichment, so that a change in the
feature table cannot silently change the target construct.


In [4]:
qa_flags = []

# --- 2a. out-of-scale Likert values in the Citizens instrument
LIK_CIT = [c for c in CIT.columns if c.startswith(("Q2.", "Q3.", "Q4.")) and
           pd.api.types.is_numeric_dtype(CIT[c])]
for c in LIK_CIT:
    bad = CIT[(CIT[c] > 5) | (CIT[c] < 1)][c].dropna()
    for idx, v in bad.items():
        qa_flags.append({"check": "out_of_scale", "instrument": "Citizens",
                         "row": int(idx), "field": c, "value": float(v),
                         "action": "set to missing"})

# --- 2b. coordinates outside the study region
LON_RANGE, LAT_RANGE = (106.60, 106.90), (10.75, 10.95)
bad_xy = CIT[~((CIT["Long"].between(*LON_RANGE)) & (CIT["Lat"].between(*LAT_RANGE)))]
for idx, r in bad_xy.iterrows():
    qa_flags.append({"check": "coordinate_out_of_region", "instrument": "Citizens",
                     "row": int(idx), "field": "Lat/Long",
                     "value": f"{r['Lat']},{r['Long']}", "action": "drop from spatial analysis"})

# --- 2c. anomalous missingness (a field missing for >40% but not a branch field)
miss = CIT.isna().mean()
for c, m in miss[miss > 0.40].items():
    qa_flags.append({"check": "high_missingness", "instrument": "Citizens", "row": -1,
                     "field": c, "value": round(float(m), 3), "action": "review (branch field?)"})

QA = pd.DataFrame(qa_flags)
print(f"Layer-2 automated checks raised {len(QA)} flags:")
if len(QA):
    print(QA.groupby("check").size().to_string())
save_table(QA, "S2_layer2_qa_flags", "Automated quality-assurance flags raised by pipeline Layer 2.")
rec("qa_flags_total", int(len(QA)))
rec("qa_out_of_scale", int((QA["check"] == "out_of_scale").sum()) if len(QA) else 0)
rec("qa_bad_coords", int((QA["check"] == "coordinate_out_of_region").sum()) if len(QA) else 0)


Layer-2 automated checks raised 4 flags:
check
coordinate_out_of_region    1
high_missingness            1
out_of_scale                2
  -> Tables/S2_layer2_qa_flags.csv  (4 rows)


1

## Step 3 — Layer 3: Enrichment

The harmonised survey records are joined to the open-data feature layer verified in
Paper 1 (`Paper1/Results/master_table.csv`): 539 geolocated respondents x 37 features
derived from Copernicus DEM GLO-30, Sentinel-1 SAR, Sentinel-2 MSI, Landsat 8/9 LST,
GHSL POP/BUILT, JRC Global Surface Water and OpenStreetMap.

Reuse rather than re-download is a **design decision**, not a convenience: the paper's
claim is that a working decision engine can be built from freely available sources, and
the strongest demonstration of that is that the same download package serves four papers.


In [5]:
MT = pd.read_csv(P1RES / "master_table.csv")
MT["is_resident"] = (MT["group"] == "Residents").astype(int)
print(f"enriched table: {MT.shape[0]} respondents x {MT.shape[1]} columns")
print(MT["group"].value_counts().to_string())

BURDEN_ITEMS = ["b_air", "b_flood", "b_noise", "b_water", "b_congest", "b_accident"]
DEMOG = ["female", "age", "edu", "income"]

# --- Layer-2 re-verification AFTER enrichment (the self-check the paper describes) ----
def cronbach_alpha(df):
    df = df.dropna()
    k = df.shape[1]
    return k / (k - 1) * (1 - df.var(ddof=1).sum() / df.sum(axis=1).var(ddof=1))

RES_ONLY = MT[MT.group == "Residents"].copy()
alpha = cronbach_alpha(RES_ONLY[BURDEN_ITEMS])
print(f"\nLayer-2 re-check | Cronbach's alpha (burden, residents) = {alpha:.3f}")
rec("alpha_burden", round(float(alpha), 3))

# KMO
def kmo(df):
    df = df.dropna()
    corr = np.corrcoef(df.values, rowvar=False)
    inv = np.linalg.pinv(corr)
    d = np.sqrt(np.diag(inv))
    pcorr = -inv / np.outer(d, d)
    np.fill_diagonal(pcorr, 0); np.fill_diagonal(corr, 0)
    return (corr ** 2).sum() / ((corr ** 2).sum() + (pcorr ** 2).sum())

LIKERT21 = BURDEN_ITEMS + ["i_drain", "i_facil", "i_road",
                           "p_security", "p_change", "p_advantage", "p_green"]
LIKERT21 = [c for c in LIKERT21 if c in MT.columns]
kmo_val = kmo(RES_ONLY[LIKERT21])
print(f"Layer-2 re-check | KMO ({len(LIKERT21)} items)          = {kmo_val:.3f}")
rec("kmo", round(float(kmo_val), 3))

# Moran's I of the target
from libpysal.weights import KNN
from esda.moran import Moran

coords_res = RES_ONLY[["x", "y"]].values if "x" in RES_ONLY else None
W = KNN.from_array(coords_res, k=8)
W.transform = "r"
mi_target = Moran(RES_ONLY["burden_mean"].values, W, permutations=999)
mi_flood  = Moran(RES_ONLY["b_flood"].values, W, permutations=999)
print(f"Layer-2 re-check | Moran's I burden_mean = {mi_target.I:+.4f} (p = {mi_target.p_sim:.4f})")
print(f"Layer-2 re-check | Moran's I b_flood     = {mi_flood.I:+.4f} (p = {mi_flood.p_sim:.4f})")
rec("moran_burden_mean", round(float(mi_target.I), 4))
rec("moran_burden_mean_p", float(mi_target.p_sim))
rec("moran_flood", round(float(mi_flood.I), 4))
rec("moran_flood_p", float(mi_flood.p_sim))

QA_RECHECK = pd.DataFrame([
    {"statistic": "Cronbach's alpha (6 burden items)", "value": f"{alpha:.3f}",
     "benchmark_paper1": "0.846", "status": "PASS" if abs(alpha - 0.846) < 0.01 else "CHECK"},
    {"statistic": f"KMO ({len(LIKERT21)} Likert items)", "value": f"{kmo_val:.3f}",
     "benchmark_paper1": "0.843", "status": "PASS" if abs(kmo_val - 0.843) < 0.05 else "CHECK"},
    {"statistic": "Moran's I, flooding (KNN k=8)", "value": f"{mi_flood.I:+.4f}",
     "benchmark_paper1": "+0.1834", "status": "PASS" if abs(mi_flood.I - 0.1834) < 0.01 else "CHECK"},
    {"statistic": "Geolocated respondents", "value": str(len(MT)),
     "benchmark_paper1": "539", "status": "PASS" if len(MT) == 539 else "CHECK"},
])
print()
print(QA_RECHECK.to_string(index=False))
save_table(QA_RECHECK, "S3_layer2_recheck",
           "Layer-2 re-verification of construct statistics after enrichment.")


enriched table: 539 respondents x 76 columns
group
Residents    459
SMEs          80

Layer-2 re-check | Cronbach's alpha (burden, residents) = 0.846
Layer-2 re-check | KMO (13 items)          = 0.841


Layer-2 re-check | Moran's I burden_mean = +0.1402 (p = 0.0010)
Layer-2 re-check | Moran's I b_flood     = +0.1834 (p = 0.0010)

                        statistic   value benchmark_paper1 status
Cronbach's alpha (6 burden items)   0.846            0.846   PASS
            KMO (13 Likert items)   0.841            0.843   PASS
    Moran's I, flooding (KNN k=8) +0.1834          +0.1834   PASS
           Geolocated respondents     539              539   PASS
  -> Tables/S3_layer2_recheck.csv  (4 rows)


,statistic,value,benchmark_paper1,status
0,Cronbach's alpha (6 burden items),0.846,0.846,PASS
1,KMO (13 Likert items),0.841,0.843,PASS
2,"Moran's I, flooding (KNN k=8)",+0.1834,+0.1834,PASS
3,Geolocated respondents,539,539,PASS


## Step 4 — RQ1: Diagnosing the barriers to digital urban governance

Three independent readings of the same question:

1. **Q2.10** — experts' unprompted top-3 digital-transformation barriers (frequency + co-occurrence network)
2. **Q3.xA1–A5** — a forced-priority matrix of 5 barrier classes across 6 urban issues (6 x 5 x 41)
3. **Authorities Q2.3** — the same barrier classes, asked of the institutions themselves

The matrix in (2) is the most informative structure in the Experts instrument and has
not previously been analysed.


In [6]:
from scipy import stats

# ---------------------------------------------------------------- 4.1 Q2.10 frequencies
def split_multi(series):
    """Split a Google-Forms multi-select cell into a list of clean options."""
    out = []
    for v in series.dropna().astype(str):
        parts = [p.strip() for p in re.split(r",(?![^(]*\))", v) if p.strip()]
        out.append(parts)
    return out

exp_dt = split_multi(EXP["Q2.10"])
n_dt = len(exp_dt)
flat = [b for row in exp_dt for b in row]
freq = pd.Series(flat).value_counts()

rows = []
for label, n in freq.items():
    en, cls = DT_BARRIERS.get(label, (label, "OTHER"))
    p = n / n_dt
    lo, hi = stats.beta.ppf([0.025, 0.975], n + 0.5, n_dt - n + 0.5)   # Jeffreys interval
    rows.append({"barrier_vi": label, "barrier_en": en, "class": cls, "n": int(n),
                 "pct": round(100 * p, 1), "ci_low": round(100 * lo, 1), "ci_high": round(100 * hi, 1)})
T1 = pd.DataFrame(rows)
print(f"Experts answering Q2.10: n = {n_dt} (each selects 3)")
print(T1.to_string(index=False))
save_table(T1, "T1_expert_dt_barriers",
           "Barriers to digital transformation identified by experts (Q2.10, choose 3, n=41).")
rec("n_experts_q210", n_dt)
for _, r in T1.iterrows():
    rec(f"barrier_pct::{r.barrier_en}", r.pct)

# Institutional share of all selections
inst_sel = sum(1 for b in flat if DT_BARRIERS.get(b, ("", "OTHER"))[1] == "INSTITUTIONAL")
tot_sel = len(flat)
p_inst = inst_sel / tot_sel
lo, hi = stats.beta.ppf([0.025, 0.975], inst_sel + 0.5, tot_sel - inst_sel + 0.5)
print(f"\nInstitutional share of all {tot_sel} selections: "
      f"{100*p_inst:.1f}% [{100*lo:.1f}, {100*hi:.1f}]")
rec("inst_share_pct", round(100 * p_inst, 1))
rec("inst_share_ci", [round(100 * lo, 1), round(100 * hi, 1)])
rec("n_selections_total", tot_sel)

# The decisive negative: no purely technical barrier offered OR volunteered
tech_terms = ["thuật toán", "mô hình", "tính toán", "phần mềm", "công nghệ", "kỹ thuật"]
volunteered = [b for b in flat if b not in DT_BARRIERS]
tech_volunteered = [b for b in volunteered if any(t in b.lower() for t in tech_terms)]
print(f"Barriers volunteered outside the option list: {len(volunteered)} "
      f"| of which algorithmic/technical: {len(tech_volunteered)}")
rec("n_volunteered_barriers", len(volunteered))
rec("n_technical_volunteered", len(tech_volunteered))

# ---------------------------------------------------------------- 4.2 co-occurrence network
import networkx as nx
labels = [l for l in freq.index if l in DT_BARRIERS]
idx = {l: i for i, l in enumerate(labels)}
C = np.zeros((len(labels), len(labels)), dtype=int)
for row in exp_dt:
    present = [b for b in row if b in idx]
    for a, b in itertools.combinations(sorted(set(present)), 2):
        C[idx[a], idx[b]] += 1; C[idx[b], idx[a]] += 1

CO = pd.DataFrame(C, index=[DT_BARRIERS[l][0] for l in labels],
                     columns=[DT_BARRIERS[l][0] for l in labels])
print("\nCo-occurrence matrix (counts of joint selection):")
print(CO.to_string())
save_table(CO.reset_index().rename(columns={"index": "barrier"}), "T2_barrier_cooccurrence",
           "Co-occurrence counts of digital-transformation barriers (Experts Q2.10).")

# Jaccard-weighted graph + observed-vs-expected under independence
edge_rows = []
for a, b in itertools.combinations(labels, 2):
    obs = C[idx[a], idx[b]]
    na, nb = freq[a], freq[b]
    # expected joint selections under random choice of 3 from the option set
    exp_joint = na * nb * 3 * 2 / (n_dt * len(labels) * (len(labels) - 1)) * (len(labels) - 1) / 2
    exp_joint = na * nb / n_dt * (2 / (len(labels) - 1)) * (len(labels) - 1) / 2
    jac = obs / (na + nb - obs) if (na + nb - obs) else 0
    edge_rows.append({"a": DT_BARRIERS[a][0], "b": DT_BARRIERS[b][0],
                      "observed": int(obs), "jaccard": round(jac, 3),
                      "class_pair": f"{DT_BARRIERS[a][1]}-{DT_BARRIERS[b][1]}"})
EDGES = pd.DataFrame(edge_rows).sort_values("observed", ascending=False)
print("\nStrongest barrier pairings:")
print(EDGES.head(6).to_string(index=False))
save_table(EDGES, "T3_barrier_edges", "Barrier co-occurrence edge list with Jaccard weights.")

G = nx.Graph()
for l in labels:
    G.add_node(DT_BARRIERS[l][0], n=int(freq[l]), cls=DT_BARRIERS[l][1])
for _, e in EDGES.iterrows():
    if e.observed > 0:
        G.add_edge(e.a, e.b, weight=int(e.observed), jaccard=e.jaccard)
print(f"\nnetwork: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, "
      f"density {nx.density(G):.3f}")
rec("network_density", round(float(nx.density(G)), 3))

# Do institutional barriers cluster together more than chance?
inst_nodes = [n for n, d in G.nodes(data=True) if d["cls"] == "INSTITUTIONAL"]
w_within = sum(d["weight"] for u, v, d in G.edges(data=True)
               if u in inst_nodes and v in inst_nodes)
w_total = sum(d["weight"] for _, _, d in G.edges(data=True))
obs_ratio = w_within / w_total
# permutation: reshuffle the class labels over nodes
rng = np.random.default_rng(RNG)
node_list = list(G.nodes())
cls_list = [G.nodes[n]["cls"] for n in node_list]
null = []
for _ in range(N_PERM):
    perm = rng.permutation(cls_list)
    lab = dict(zip(node_list, perm))
    w = sum(d["weight"] for u, v, d in G.edges(data=True)
            if lab[u] == "INSTITUTIONAL" and lab[v] == "INSTITUTIONAL")
    null.append(w / w_total)
p_clust = (np.sum(np.array(null) >= obs_ratio) + 1) / (N_PERM + 1)
print(f"institutional-institutional edge weight share = {obs_ratio:.3f} "
      f"(permutation p = {p_clust:.4f}) [6 nodes - underpowered, reported as descriptive]")
rec("inst_edge_share", round(float(obs_ratio), 3))
rec("inst_edge_share_p", round(float(p_clust), 4))
top3 = EDGES.head(3)
rec("top3_edges_all_institutional",
    bool((top3.class_pair == "INSTITUTIONAL-INSTITUTIONAL").all()))
rec("top3_edge_weight_sum", int(top3.observed.sum()))
print(f"the three heaviest edges carry {top3.observed.sum()} of {tot_sel} selections and are "
      f"{'all' if RESULTS['top3_edges_all_institutional'] else 'not all'} institutional-institutional")

# ---------------------------------------------------------------- 4.3 selection vs chance
# The option list is not class-balanced (4 of 6 options are institutional), so the raw
# institutional share cannot by itself establish institutional dominance. The
# well-identified test is per-option: under a null in which each expert picks 3 of the 6
# options at random, every option is selected by 50% of experts.
sel_rows = []
for label, n in freq.items():
    if label not in DT_BARRIERS:
        continue
    p_bin = stats.binomtest(int(n), n_dt, 0.5).pvalue
    sel_rows.append({"barrier_en": DT_BARRIERS[label][0], "class": DT_BARRIERS[label][1],
                     "n": int(n), "pct": round(100 * n / n_dt, 1),
                     "expected_pct_if_random": 50.0,
                     "p_binomial": round(float(p_bin), 4)})
SEL = pd.DataFrame(sel_rows)
SEL["p_holm"] = stats.false_discovery_control(SEL.p_binomial.values, method="bh").round(4)
print("\nSelection frequency against a uniform-choice null (each option expected at 50%):")
print(SEL.to_string(index=False))
save_table(SEL, "T4_selection_vs_chance",
           "Expert barrier selection tested against a uniform choose-3-of-6 null.")
above = SEL[(SEL.pct > 50) & (SEL.p_holm < 0.05)]
below = SEL[(SEL.pct < 50) & (SEL.p_holm < 0.05)]
print(f"\nselected above chance (BH q<.05): {list(above.barrier_en)}")
print(f"selected below chance (BH q<.05): {list(below.barrier_en)}")
rec("above_chance_barriers", list(above.barrier_en))
rec("above_chance_all_institutional", bool((above["class"] == "INSTITUTIONAL").all()) and len(above) > 0)
rec("below_chance_barriers", list(below.barrier_en))


Experts answering Q2.10: n = 41 (each selects 3)
                         barrier_vi                             barrier_en         class  n  pct  ci_low  ci_high
       Cơ chế phối hợp giữa các bên   Inter-agency coordination mechanisms INSTITUTIONAL 31 75.6    61.0     86.7
            Chính sách chưa rõ ràng                         Unclear policy INSTITUTIONAL 26 63.4    48.2     76.8
                   Tiếp cận dữ liệu                            Data access INSTITUTIONAL 25 61.0    45.7     74.7
                    Thiếu ngân sách                    Insufficient budget      RESOURCE 17 41.5    27.4     56.7
Chuyên môn khác nhau giữa ban ngành Divergent expertise across departments INSTITUTIONAL 13 31.7    19.1     46.8
              Thiếu người thực hiện                 Insufficient personnel      RESOURCE 11 26.8    15.2     41.6
  -> Tables/T1_expert_dt_barriers.csv  (6 rows)

Institutional share of all 123 selections: 77.2% [69.2, 84.0]
Barriers volunteered outside the option li

['Insufficient personnel']

### 4.3 — The barrier matrix (Experts Q3.xA1–A5)

For each of six urban issues, experts scored five barrier classes. This structure is the
well-identified counterpart to Q2.10: whereas the Q2.10 option list contains four
institutional options out of six, the matrix presents each of the five classes exactly
once per issue, so the classes can be compared directly.

**Instrument note.** The paper questionnaire asked respondents to *rank* the five classes.
In the delivered form each class was rendered as an independent 1–5 scale, and only
2–3 of ~32 respondents per issue produced a complete permutation. The matrix is therefore
analysed as **five repeated 1–5 importance ratings**, not as a ranking. This is stated in
the manuscript; treating it as a ranking would be wrong.


In [7]:
mat_rows = []
for iss, q in EXP_ISSUE_Q.items():
    for a, (label_en, cls) in BARRIER5.items():
        col = f"{q}{a}"
        if col not in EXP.columns:
            continue
        v = pd.to_numeric(EXP[col], errors="coerce").replace(0, np.nan)
        for rid, val in v.items():
            mat_rows.append({"expert": rid, "issue": iss, "issue_en": ISSUE_EN[iss],
                             "barrier_code": a, "barrier_en": label_en, "class": cls,
                             "rating": val})
MATRIX = pd.DataFrame(mat_rows)
MATRIX_v = MATRIX.dropna(subset=["rating"])
print(f"barrier matrix: {MATRIX_v.expert.nunique()} experts x {MATRIX_v.issue.nunique()} issues "
      f"x {MATRIX_v.barrier_code.nunique()} classes = {len(MATRIX_v)} ratings")
rec("matrix_n_ratings", int(len(MATRIX_v)))
rec("matrix_n_experts", int(MATRIX_v.expert.nunique()))

# ---- mean rating per class per issue
piv = MATRIX_v.pivot_table(index="issue_en", columns="barrier_en", values="rating", aggfunc="mean")
piv = piv[[BARRIER5[a][0] for a in ["A1", "A2", "A3", "A4", "A5"]]]
print("\nmean barrier importance (1-5) by urban issue:")
print(piv.round(2).to_string())
T5 = piv.round(3).reset_index()
save_table(T5, "T5_barrier_matrix_means",
           "Mean expert-rated importance of five barrier classes across six urban issues (Q3.xA).")

# ---- overall class means with bootstrap CIs
rng = np.random.default_rng(RNG)
cls_rows = []
for a in ["A1", "A2", "A3", "A4", "A5"]:
    lab, cls = BARRIER5[a]
    v = MATRIX_v[MATRIX_v.barrier_code == a].rating.values
    boot = [rng.choice(v, size=len(v), replace=True).mean() for _ in range(N_BOOT)]
    cls_rows.append({"barrier_code": a, "barrier_en": lab, "class": cls,
                     "n_ratings": len(v), "mean": round(v.mean(), 3),
                     "sd": round(v.std(ddof=1), 3),
                     "ci_low": round(np.percentile(boot, 2.5), 3),
                     "ci_high": round(np.percentile(boot, 97.5), 3)})
T6 = pd.DataFrame(cls_rows).sort_values("mean", ascending=False)
print("\noverall barrier-class importance (pooled over issues):")
print(T6.to_string(index=False))
save_table(T6, "T6_barrier_class_overall",
           "Pooled importance of the five barrier classes with bootstrap 95% CIs.")
for _, r in T6.iterrows():
    rec(f"matrix_mean::{r.barrier_en}", r["mean"])

# ---- Friedman test within each issue (complete cases only)
fried_rows = []
for iss, q in EXP_ISSUE_Q.items():
    cols = [f"{q}{a}" for a in ["A1", "A2", "A3", "A4", "A5"]]
    sub = EXP[cols].apply(pd.to_numeric, errors="coerce").replace(0, np.nan).dropna()
    if len(sub) < 8:
        continue
    stat, p = stats.friedmanchisquare(*[sub[c].values for c in cols])
    kendall_w = stat / (len(sub) * (5 - 1))
    fried_rows.append({"issue_en": ISSUE_EN[iss], "n_complete": len(sub),
                       "friedman_chi2": round(float(stat), 3), "df": 4,
                       "p": float(p), "kendall_w": round(float(kendall_w), 3)})
FRIED = pd.DataFrame(fried_rows)
FRIED["p_bh"] = stats.false_discovery_control(FRIED.p.values, method="bh")
FRIED["p"] = FRIED.p.round(4); FRIED["p_bh"] = FRIED.p_bh.round(4)
print("\nFriedman test: do the five barrier classes differ within an issue?")
print(FRIED.to_string(index=False))
save_table(FRIED, "T7_friedman_by_issue",
           "Friedman tests of barrier-class differences within each urban issue.")
rec("friedman_significant_issues", int((FRIED.p_bh < 0.05).sum()))
rec("friedman_n_issues", int(len(FRIED)))

# ---- the decisive contrast: technical vs institutional, paired within expert-issue
pairs = MATRIX_v.pivot_table(index=["expert", "issue"], columns="barrier_code", values="rating")
pairs = pairs.dropna(subset=["A1", "A4"])
d = pairs["A4"] - pairs["A1"]
w_stat, w_p = stats.wilcoxon(pairs["A4"], pairs["A1"])
t_stat, t_p = stats.ttest_rel(pairs["A4"], pairs["A1"])
dz = d.mean() / d.std(ddof=1)
boot = [rng.choice(d.values, size=len(d), replace=True).mean() for _ in range(N_BOOT)]
print(f"\nPaired contrast, institutional (A4) minus technical (A1), n = {len(d)} expert-issue pairs")
print(f"  mean difference = {d.mean():+.3f} "
      f"[{np.percentile(boot,2.5):+.3f}, {np.percentile(boot,97.5):+.3f}]")
print(f"  Wilcoxon W = {w_stat:.1f}, p = {w_p:.4g} | paired t = {t_stat:.3f}, p = {t_p:.4g} "
      f"| Cohen's dz = {dz:+.3f}")
rec("inst_minus_tech_mean", round(float(d.mean()), 3))
rec("inst_minus_tech_ci", [round(float(np.percentile(boot, 2.5)), 3),
                           round(float(np.percentile(boot, 97.5)), 3)])
rec("inst_minus_tech_p", float(w_p))
rec("inst_minus_tech_dz", round(float(dz), 3))
rec("inst_minus_tech_n", int(len(d)))

# ---- rank of each class within each expert-issue profile
# rank 1 = most important. Ties are frequent (many experts give several classes the same
# score), so "ranked first" is credited fractionally across the tied classes rather than
# awarded to whichever column happens to come first.
AC = ["A1", "A2", "A3", "A4", "A5"]
prof = MATRIX_v.pivot_table(index=["expert", "issue"], columns="barrier_code", values="rating").dropna()
ranks = prof[AC].rank(axis=1, ascending=False, method="average")   # 1 = most important
is_max = prof[AC].eq(prof[AC].max(axis=1), axis=0)
first_share = is_max.div(is_max.sum(axis=1), axis=0)               # fractional credit
n_tied = int((is_max.sum(axis=1) > 1).sum())
print(f"\nwithin-profile ranking over {len(prof)} complete expert-issue profiles "
      f"({n_tied} of them, {100*n_tied/len(prof):.0f}%, have a tie for the top class)")
RANKSUM = pd.DataFrame({
    "barrier_en": [BARRIER5[a][0] for a in AC],
    "class":      [BARRIER5[a][1] for a in AC],
    "mean_rating": [round(float(prof[a].mean()), 2) for a in AC],
    "mean_rank":  [round(float(ranks[a].mean()), 2) for a in AC],
    "pct_top_fractional": [round(float(100 * first_share[a].mean()), 1) for a in AC],
}).sort_values("mean_rank")
print(RANKSUM.to_string(index=False))
save_table(RANKSUM, "T8_barrier_within_profile_rank",
           "Within-profile rank of each barrier class across complete expert-issue profiles.")
rec("tech_mean_rank", round(float(ranks["A1"].mean()), 2))
rec("tech_top_pct", round(float(100 * first_share["A1"].mean()), 1))
rec("n_complete_profiles", int(len(prof)))
rec("pct_profiles_with_top_tie", round(100 * n_tied / len(prof), 1))
rec("matrix_worst_class", RANKSUM.iloc[-1].barrier_en)
rec("matrix_best_class", RANKSUM.iloc[0].barrier_en)
rec("matrix_best_class_share", float(RANKSUM.iloc[0].pct_top_fractional))

# ---- is the technical class ever the modal leader?
lead = RANKSUM.set_index("barrier_en")["pct_top_fractional"]
print(f"\ntechnical class leads {lead[BARRIER5['A1'][0]]:.1f}% of profiles "
      f"vs {lead.max():.1f}% for the leading class ({lead.idxmax()})")

# ---- honest summary of what the matrix does and does not establish
print("""
READING OF THE MATRIX
  The five classes are rated close to the scale midpoint and the Friedman tests do not
  reject the null of equal importance within any issue (all BH q > .40, Kendall's W <= .05).
  The matrix therefore does NOT independently establish institutional dominance.
  What it does establish is a consistent ORDERING in which the technical class is never
  the leader, and a striking absence of differentiation: at the level of substantive urban
  problems, experts treat every class of barrier as moderately binding at once.
  This is reported as such; it is not evidence for the Q2.10 result, it is a second,
  weaker and differently-shaped observation about the same system.""")


barrier matrix: 33 experts x 6 issues x 5 classes = 949 ratings

mean barrier importance (1-5) by urban issue:
barrier_en          Technical / expertise / natural conditions  Financial  High-quality human resources  Policy and legal framework  Social characteristics (awareness, culture)
issue_en                                                                                                                                                                        
Air pollution                                             3.03       3.13                          2.91                        2.88                                         3.12
Flooding                                                  2.90       3.06                          3.16                        3.12                                         3.35
Noise pollution                                           2.77       2.78                          3.10                        3.31                                         3.25
Traf

  -> Tables/T8_barrier_within_profile_rank.csv  (5 rows)

technical class leads 15.4% of profiles vs 27.2% for the leading class (Social characteristics (awareness, culture))

READING OF THE MATRIX
  The five classes are rated close to the scale midpoint and the Friedman tests do not
  reject the null of equal importance within any issue (all BH q > .40, Kendall's W <= .05).
  The matrix therefore does NOT independently establish institutional dominance.
  What it does establish is a consistent ORDERING in which the technical class is never
  the leader, and a striking absence of differentiation: at the level of substantive urban
  problems, experts treat every class of barrier as moderately binding at once.
  This is reported as such; it is not evidence for the Q2.10 result, it is a second,
  weaker and differently-shaped observation about the same system.


### 4.4 — Experts against the institutions they assess

Two comparisons: the assessment of digital-transformation progress (Experts Q2.9 vs
Authorities Q2.16, identical wording and scale), and the barriers to solving urban
problems (Experts Q2.3 vs Authorities Q2.3, identical option list).


In [8]:
# ---- 4.4a assessment of digital-transformation progress
e29 = EXP["Q2.9"].map(SCALE_EFFECT).dropna()
a216 = AUT["Q2.16"].map(SCALE_EFFECT).dropna()
u, p_u = stats.mannwhitneyu(e29, a216, alternative="two-sided")
# rank-biserial effect size
rb = 1 - 2 * u / (len(e29) * len(a216))
print(f"Digital-transformation progress (1 = very difficult ... 5 = very effective)")
print(f"  Experts     n = {len(e29)}, mean = {e29.mean():.2f}, median = {e29.median():.0f}")
print(f"  Authorities n = {len(a216)}, mean = {a216.mean():.2f}, median = {a216.median():.0f}")
print(f"  Mann-Whitney U = {u:.1f}, p = {p_u:.4f}, rank-biserial r = {rb:+.3f}")
rec("dt_progress_expert_mean", round(float(e29.mean()), 2))
rec("dt_progress_auth_mean", round(float(a216.mean()), 2))
rec("dt_progress_p", round(float(p_u), 4))
rec("dt_progress_rb", round(float(rb), 3))

# ordered categorical distribution
dist = pd.DataFrame({
    "Experts (Q2.9)": EXP["Q2.9"].value_counts(),
    "Authorities (Q2.16)": AUT["Q2.16"].value_counts()}).fillna(0).astype(int)
order = ["Rất khó khăn", "Còn nhiều khó khăn", "Trung bình", "Hiệu quả", "Rất hiệu quả"]
dist = dist.reindex([o for o in order if o in dist.index])
dist.index = ["Very difficult", "Considerable difficulty", "Average", "Effective", "Very effective"][:len(dist)]
dist["Experts %"] = (100 * dist["Experts (Q2.9)"] / dist["Experts (Q2.9)"].sum()).round(1)
dist["Authorities %"] = (100 * dist["Authorities (Q2.16)"] / dist["Authorities (Q2.16)"].sum()).round(1)
print("\n" + dist.to_string())
save_table(dist.reset_index().rename(columns={"index": "assessment"}), "T9_dt_progress_gap",
           "Assessment of digital-transformation progress: experts vs authorities.")

pos_e = (e29 >= 4).mean(); pos_a = (a216 >= 4).mean()
ct = np.array([[(e29 >= 4).sum(), (e29 < 4).sum()], [(a216 >= 4).sum(), (a216 < 4).sum()]])
_, p_f = stats.fisher_exact(ct)
print(f"\n'effective or better': experts {100*pos_e:.1f}% vs authorities {100*pos_a:.1f}% "
      f"(Fisher exact p = {p_f:.4f})")
rec("dt_positive_expert_pct", round(100 * float(pos_e), 1))
rec("dt_positive_auth_pct", round(100 * float(pos_a), 1))
rec("dt_positive_fisher_p", round(float(p_f), 4))

# ---- 4.4b barriers to solving urban problems, same option list in both instruments
AUTH_BARRIERS = {
    "Khó khăn về kỹ thuật / chuyên môn / điều kiện tự nhiên": ("Technical / expertise / natural conditions", "TECHNICAL"),
    "Khó khăn về tài chính":                                  ("Financial", "RESOURCE"),
    "Khó khăn về nguồn nhân lực chất lượng cao":              ("High-quality human resources", "RESOURCE"),
    "Khó khăn về cơ chế chính sách / pháp luật":              ("Policy and legal framework", "INSTITUTIONAL"),
    "Khó khăn về đặc điểm xã hội (Ý thức / văn hóa / lịch sử)": ("Social characteristics", "SOCIAL"),
    "Khó khăn về đặc điểm xã hội (Ý thức / văn hóa / lịch sử) của người dân": ("Social characteristics", "SOCIAL"),
}
def barrier_freq(series, mapping):
    rows = split_multi(series)
    n = len(rows)
    flat = [b for r in rows for b in r]
    out = {}
    for b in flat:
        key = None
        for k, v in mapping.items():
            if b.strip() == k.strip():
                key = v[0]; break
        if key is None:
            key = "Other / volunteered"
        out[key] = out.get(key, 0) + 1
    return n, out

ne, fe = barrier_freq(EXP["Q2.3"], AUTH_BARRIERS)
na, fa = barrier_freq(AUT["Q2.3"], AUTH_BARRIERS)
keys = [v[0] for v in AUTH_BARRIERS.values()]
keys = list(dict.fromkeys(keys)) + ["Other / volunteered"]
rows = []
for k in keys:
    ce, ca = fe.get(k, 0), fa.get(k, 0)
    _, pf = stats.fisher_exact([[ce, ne - ce], [ca, na - ca]])
    rows.append({"barrier_en": k,
                 "class": next((v[1] for v in AUTH_BARRIERS.values() if v[0] == k), "OTHER"),
                 "experts_n": ce, "experts_pct": round(100 * ce / ne, 1),
                 "auth_n": ca, "auth_pct": round(100 * ca / na, 1),
                 "gap_pp": round(100 * ce / ne - 100 * ca / na, 1),
                 "fisher_p": round(float(pf), 4)})
T10 = pd.DataFrame(rows)
T10["p_bh"] = stats.false_discovery_control(T10.fisher_p.values, method="bh").round(4)
print(f"\nBarriers to solving urban problems (Q2.3, choose 3): experts n={ne}, authorities n={na}")
print(T10.to_string(index=False))
save_table(T10, "T10_q23_barrier_gap",
           "Barriers to solving urban problems: experts vs authorities (Q2.3, choose 3).")
rec("q23_experts_n", ne); rec("q23_auth_n", na)
sig = T10[T10.p_bh < 0.05]
rec("q23_significant_gaps", list(sig.barrier_en))
print(f"\nsignificant gaps after BH correction: "
      f"{list(sig.barrier_en) if len(sig) else 'none'}")

# ---- 4.4c: do either group have an instrument to evaluate what they build?
e213 = EXP["Q2.13"].map(SCALE_AGREE).dropna()      # "effective tools exist to evaluate projects"
a221 = AUT["Q2.21"].map(SCALE_AGREE).dropna()      # "tools exist to track investment outcomes"
u2, p2 = stats.mannwhitneyu(e213, a221, alternative="two-sided")
print(f"\nExistence of tools to evaluate projects / investments (1 = strongly disagree ... 5 = strongly agree)")
print(f"  Experts Q2.13     n = {len(e213)}, mean = {e213.mean():.2f}, agree-or-better = {100*(e213>=4).mean():.1f}%")
print(f"  Authorities Q2.21 n = {len(a221)}, mean = {a221.mean():.2f}, agree-or-better = {100*(a221>=4).mean():.1f}%")
print(f"  Mann-Whitney U = {u2:.1f}, p = {p2:.4f}")
rec("eval_tools_expert_mean", round(float(e213.mean()), 2))
rec("eval_tools_auth_mean", round(float(a221.mean()), 2))
rec("eval_tools_expert_agree_pct", round(100 * float((e213 >= 4).mean()), 1))
rec("eval_tools_auth_agree_pct", round(100 * float((a221 >= 4).mean()), 1))
rec("eval_tools_p", round(float(p2), 4))


Digital-transformation progress (1 = very difficult ... 5 = very effective)
  Experts     n = 41, mean = 2.93, median = 3
  Authorities n = 23, mean = 3.43, median = 3
  Mann-Whitney U = 322.0, p = 0.0261, rank-biserial r = +0.317

                         Experts (Q2.9)  Authorities (Q2.16)  Experts %  Authorities %
Very difficult                        1                    0        2.4            0.0
Considerable difficulty              12                    2       29.3            8.7
Average                              18                   11       43.9           47.8
Effective                             9                    8       22.0           34.8
Very effective                        1                    2        2.4            8.7
  -> Tables/T9_dt_progress_gap.csv  (5 rows)

'effective or better': experts 24.4% vs authorities 43.5% (Fisher exact p = 0.1607)

Barriers to solving urban problems (Q2.3, choose 3): experts n=41, authorities n=23
                               

0.0033

### Figure style


In [9]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import matplotlib.patheffects as pe

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 400, "savefig.bbox": "tight",
    "font.family": "DejaVu Sans", "font.size": 9,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.axisbelow": True, "axes.titlesize": 10, "axes.titleweight": "bold",
    "legend.frameon": False,
})

CLS_COLOR = {"INSTITUTIONAL": "#1b4965", "RESOURCE": "#c26a3d",
             "TECHNICAL": "#6b8f71", "SOCIAL": "#8a6ea8", "OTHER": "#9aa0a6"}
ACCENT, MUTED, INK = "#1b4965", "#9aa0a6", "#22252a"


def wrap(s, w=22):
    out, line = [], ""
    for word in s.split():
        if len(line) + len(word) + 1 > w:
            out.append(line); line = word
        else:
            line = f"{line} {word}".strip()
    out.append(line)
    return "\n".join(out)


### Figure 2 — Barrier diagnosis


In [10]:
fig = plt.figure(figsize=(13.5, 5.6))
gs = fig.add_gridspec(1, 3, width_ratios=[1.05, 1.15, 0.95], wspace=0.42)

# --- panel a: Q2.10 frequencies with CIs
ax = fig.add_subplot(gs[0, 0])
d = T1.sort_values("pct")
y = np.arange(len(d))
cols = [CLS_COLOR[c] for c in d["class"]]
ax.barh(y, d.pct, color=cols, height=0.66, zorder=3)
ax.errorbar(d.pct, y, xerr=[d.pct - d.ci_low, d.ci_high - d.pct], fmt="none",
            ecolor=INK, elinewidth=1.1, capsize=3, zorder=4)
ax.axvline(50, color=MUTED, ls="--", lw=1.1, zorder=2)
ax.text(50, len(d) - 0.35, " chance\n (3 of 6)", fontsize=7.5, color=MUTED, va="top")
ax.set_yticks(y); ax.set_yticklabels([wrap(s, 24) for s in d.barrier_en], fontsize=8)
ax.set_xlabel("% of experts selecting (n = 41)")
ax.set_xlim(0, 100)
ax.set_title("a  Barriers to digital transformation\n(Q2.10, choose 3)", loc="left")
for yi, v, hi in zip(y, d.pct, d.ci_high):
    ax.text(hi + 4.0, yi, f"{v:.1f}%", va="center", fontsize=7.8, color=INK)

# --- panel b: co-occurrence network, circular layout with radial labels
ax = fig.add_subplot(gs[0, 1])
node_order = list(freq.index[freq.index.isin([l for l in labels])])
node_order = [DT_BARRIERS[l][0] for l in node_order]
k = len(node_order)
ang0 = np.pi / 2
pos = {n: (np.cos(ang0 - 2 * np.pi * i / k), np.sin(ang0 - 2 * np.pi * i / k))
       for i, n in enumerate(node_order)}
ew = np.array([G[u][v]["weight"] for u, v in G.edges()])
for (u, v), w in zip(G.edges(), ew):
    x1, y1 = pos[u]; x2, y2 = pos[v]
    both_inst = G.nodes[u]["cls"] == "INSTITUTIONAL" == G.nodes[v]["cls"]
    ax.plot([x1, x2], [y1, y2], lw=0.35 + 4.6 * (w / ew.max()) ** 2.0,
            color=ACCENT if both_inst else MUTED,
            alpha=0.9 if both_inst else 0.35, zorder=1, solid_capstyle="round")
    if w >= 10:
        ax.text((x1 + x2) / 2, (y1 + y2) / 2, str(w), fontsize=7.4, ha="center", va="center",
                color=ACCENT if both_inst else "#6b7075", zorder=5, fontweight="bold",
                path_effects=[pe.withStroke(linewidth=3.0, foreground="white")])
for n, dta in G.nodes(data=True):
    x, y_ = pos[n]
    ax.scatter([x], [y_], s=50 + 14 * dta["n"], color=CLS_COLOR[dta["cls"]],
               edgecolor="white", linewidth=1.6, zorder=4)
    r = math.hypot(x, y_)
    lx, ly = x / r * 1.30, y_ / r * 1.30
    ax.text(lx, ly, wrap(n, 15), fontsize=7.2, zorder=6, color=INK,
            ha="center" if abs(x) < 0.35 else ("left" if x > 0 else "right"),
            va="center" if abs(y_) < 0.35 else ("bottom" if y_ > 0 else "top"))
ax.set_xlim(-2.15, 2.15); ax.set_ylim(-1.85, 1.85); ax.axis("off")
ax.set_title("b  Co-occurrence of selected barriers\n"
             "(node size = frequency; the three heaviest\nedges are all institutional–institutional)",
             loc="left")

# --- panel c: the balanced matrix
ax = fig.add_subplot(gs[0, 2])
d6 = T6.sort_values("mean")
y = np.arange(len(d6))
ax.barh(y, d6["mean"], color=[CLS_COLOR[c] for c in d6["class"]], height=0.62, zorder=3)
ax.errorbar(d6["mean"], y, xerr=[d6["mean"] - d6.ci_low, d6.ci_high - d6["mean"]],
            fmt="none", ecolor=INK, elinewidth=1.1, capsize=3, zorder=4)
ax.axvline(3, color=MUTED, ls="--", lw=1.1, zorder=2)
ax.set_yticks(y); ax.set_yticklabels([wrap(s, 20) for s in d6.barrier_en], fontsize=8)
ax.set_xlim(2.4, 3.7); ax.set_xlabel("mean importance (1–5)")
ax.set_title("c  Balanced barrier matrix\n(Q3.xA, 6 issues × 5 classes)", loc="left")
ax.text(0.98, 0.03, "differences n.s.\n(Friedman, all BH q > .40)", transform=ax.transAxes,
        ha="right", va="bottom", fontsize=7.2, color=MUTED, style="italic")

handles = [Patch(facecolor=CLS_COLOR[c], label=c.title()) for c in
           ["INSTITUTIONAL", "RESOURCE", "TECHNICAL", "SOCIAL"]]
fig.legend(handles=handles, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.035), fontsize=8.5)
fig.savefig(FIG / "Figure2_barrier_diagnosis.png")
plt.close(fig)
print("  -> Figures/Figure2_barrier_diagnosis.png")


  -> Figures/Figure2_barrier_diagnosis.png


### Figure 7 — the institutional gap on instruments (supports RQ1)


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0))

ax = axes[0]
cats = list(dist.index)
xe = dist["Experts %"].values; xa = dist["Authorities %"].values
y = np.arange(len(cats)); h = 0.38
ax.barh(y + h/2, xe, height=h, color=ACCENT, label=f"Experts (n={len(e29)})", zorder=3)
ax.barh(y - h/2, xa, height=h, color="#c26a3d", label=f"Authorities (n={len(a216)})", zorder=3)
ax.set_yticks(y); ax.set_yticklabels(cats, fontsize=8.5)
ax.set_xlabel("% of group")
ax.set_title("a  Progress of digital transformation\n"
             f"Mann–Whitney p = {p_u:.3f}, r = {rb:+.2f}", loc="left")
ax.legend(fontsize=8, loc="lower right")

ax = axes[1]
groups = [f"Experts\nQ2.13 (n={len(e213)})", f"Authorities\nQ2.21 (n={len(a221)})"]
vals = [100 * (e213 >= 4).mean(), 100 * (a221 >= 4).mean()]
means = [e213.mean(), a221.mean()]
bars = ax.bar(groups, vals, color=[ACCENT, "#c26a3d"], width=0.5, zorder=3)
for b, v, m in zip(bars, vals, means):
    ax.text(b.get_x() + b.get_width()/2, v + 1.6, f"{v:.1f}%\n(mean {m:.2f})",
            ha="center", fontsize=8.5, color=INK)
ax.set_ylabel("% agreeing that effective\nevaluation instruments exist")
ax.set_ylim(0, 62)
ax.set_title("b  Do instruments to evaluate\nprojects already exist?\n"
             f"Mann–Whitney p = {p2:.4f}", loc="left")
fig.tight_layout()
fig.savefig(FIG / "Figure7_institutional_gap.png")
plt.close(fig)
print("  -> Figures/Figure7_institutional_gap.png")


  -> Figures/Figure7_institutional_gap.png


## Step 5 — RQ3: Layer 4 inference, the model ladder

A disciplined three-rung ladder, chosen because n = 459 is small by machine-learning
standards. No deep learning is used anywhere in this paper.

| Model | Features | Purpose |
|---|---|---|
| M0 | demographics only (4) | baseline: what the archive alone predicts |
| M1 | demographics + open data (linear) | the increment attributable to open data |
| M2 | demographics + open data (gradient boosting) | the increment attributable to model class |

Every rung is evaluated under **two** resampling schemes: conventional random 5x5 nested
cross-validation, and **spatial block cross-validation**. On geolocated survey data the
gap between the two is the honest measure of how much of the apparent skill is spatial
leakage. Paper 1 established the gap on this corridor; this paper reproduces it as a
property of the pipeline, not of one analysis.


In [12]:
from sklearn.model_selection import KFold, GroupKFold, GridSearchCV, cross_val_predict
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.cluster import KMeans

TARGET = "burden_mean"
D = MT[MT.group == "Residents"].copy().reset_index(drop=True)
D = D.dropna(subset=[TARGET, "x", "y"]).reset_index(drop=True)

# Feature sets ---------------------------------------------------------------
F_DEMOG = ["female", "age", "edu", "income"]
F_OPEN = [c for c in OPEN_FEATURES if c in D.columns]
# drop features that are constant or near-constant on the residents subset
F_OPEN = [c for c in F_OPEN if D[c].std(skipna=True) > 1e-9 and D[c].notna().mean() > 0.90]
F_ALL = F_DEMOG + F_OPEN
print(f"training set n = {len(D)}")
print(f"  demographic features : {len(F_DEMOG)}")
print(f"  open-data features   : {len(F_OPEN)}")
rec("ml_n_train", int(len(D)))
rec("ml_n_demog_features", len(F_DEMOG))
rec("ml_n_open_features", len(F_OPEN))

# Spatial blocks --------------------------------------------------------------
N_BLOCKS = 10
km = KMeans(n_clusters=N_BLOCKS, random_state=RNG, n_init=20).fit(D[["x", "y"]].values)
D["block"] = km.labels_
print(f"spatial blocks: {N_BLOCKS}, sizes {np.bincount(D.block)}")
rec("n_spatial_blocks", N_BLOCKS)


def make_model(kind):
    if kind == "ols":
        return Pipeline([("imp", SimpleImputer(strategy="median")),
                         ("sc", StandardScaler()),
                         ("m", LinearRegression())])
    if kind == "ridge":
        return Pipeline([("imp", SimpleImputer(strategy="median")),
                         ("sc", StandardScaler()),
                         ("m", RidgeCV(alphas=np.logspace(-3, 3, 25)))])
    if kind == "gbm":
        return Pipeline([("imp", SimpleImputer(strategy="median")),
                         ("m", HistGradientBoostingRegressor(random_state=RNG))])
    if kind == "rf":
        return Pipeline([("imp", SimpleImputer(strategy="median")),
                         ("m", RandomForestRegressor(n_estimators=500, random_state=RNG,
                                                     min_samples_leaf=3, n_jobs=-1))])
    raise ValueError(kind)


GBM_GRID = {"m__max_depth": [2, 3, None],
            "m__learning_rate": [0.03, 0.06, 0.10],
            "m__max_leaf_nodes": [7, 15, 31],
            "m__min_samples_leaf": [10, 20],
            "m__l2_regularization": [0.0, 1.0]}


def nested_cv(X, y, kind, groups=None, n_out=5, n_in=5, tune=False):
    """Nested CV. Returns out-of-fold predictions and per-fold metrics."""
    if groups is None:
        outer = KFold(n_splits=n_out, shuffle=True, random_state=RNG)
        splits = outer.split(X)
    else:
        outer = GroupKFold(n_splits=n_out)
        splits = outer.split(X, y, groups)
    oof = np.full(len(y), np.nan)
    fold_metrics = []
    for k, (tr, te) in enumerate(splits):
        if tune:
            inner = KFold(n_splits=n_in, shuffle=True, random_state=RNG + k)
            gs = GridSearchCV(make_model(kind), GBM_GRID, cv=inner,
                              scoring="neg_mean_squared_error", n_jobs=-1)
            gs.fit(X.iloc[tr], y[tr])
            est = gs.best_estimator_
        else:
            est = make_model(kind).fit(X.iloc[tr], y[tr])
        p = est.predict(X.iloc[te])
        oof[te] = p
        fold_metrics.append({"fold": k, "n_test": len(te),
                             "r2": r2_score(y[te], p),
                             "mae": mean_absolute_error(y[te], p),
                             "rmse": math.sqrt(mean_squared_error(y[te], p))})
    return oof, pd.DataFrame(fold_metrics)


# ---------------------------------------------------------------- result cache
# The cross-validation loops below take roughly 8-12 minutes on a laptop. Results are
# cached so that re-running the notebook to regenerate figures or tables is fast.
# Set USE_CACHE = False (or delete Results/_cv_cache.pkl) to force a full recomputation;
# a clean recomputation is what was used to produce every number in the manuscript.
import pickle
USE_CACHE = True
CACHE_PATH = RES / "_cv_cache.pkl"
_CACHE = {}
if USE_CACHE and CACHE_PATH.exists():
    try:
        _CACHE = pickle.loads(CACHE_PATH.read_bytes())
        print(f"loaded CV cache with {len(_CACHE)} entries "
              f"(set USE_CACHE = False to recompute from scratch)")
    except Exception:
        _CACHE = {}


def cached(key, fn):
    if USE_CACHE and key in _CACHE:
        return _CACHE[key]
    val = fn()
    _CACHE[key] = val
    CACHE_PATH.write_bytes(pickle.dumps(_CACHE))
    return val


def boot_ci(y, p, metric="r2", n=N_BOOT, seed=RNG):
    rg = np.random.default_rng(seed)
    idx = np.arange(len(y)); out = []
    for _ in range(n):
        b = rg.choice(idx, size=len(idx), replace=True)
        if metric == "r2":
            out.append(r2_score(y[b], p[b]))
        elif metric == "mae":
            out.append(mean_absolute_error(y[b], p[b]))
        else:
            out.append(math.sqrt(mean_squared_error(y[b], p[b])))
    return float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))


y = D[TARGET].values
LADDER = [
    ("M0  demographics only (OLS)",        F_DEMOG, "ols",   False),
    ("M1  + open data (OLS)",              F_ALL,   "ols",   False),
    ("M1r + open data (ridge)",            F_ALL,   "ridge", False),
    ("M2  + open data (gradient boosting)", F_ALL,  "gbm",   RUN_NESTED_CV),
    ("M2r + open data (random forest)",    F_ALL,   "rf",    False),
]

lad_rows, OOF = [], {}
for name, feats, kind, tune in LADDER:
    X = D[feats]
    for scheme, groups in [("random 5-fold", None), ("spatial block", D.block.values)]:
        oof, fm = cached(f"cv|{name}|{scheme}",
                         lambda X=X, kind=kind, groups=groups, tune=tune:
                             nested_cv(X, y, kind, groups=groups, tune=tune))
        r2 = r2_score(y, oof)
        lo, hi = boot_ci(y, oof, "r2")
        mae = mean_absolute_error(y, oof)
        mlo, mhi = boot_ci(y, oof, "mae")
        lad_rows.append({"model": name, "scheme": scheme, "n_features": len(feats),
                         "r2": round(r2, 4), "r2_lo": round(lo, 4), "r2_hi": round(hi, 4),
                         "mae": round(mae, 4), "mae_lo": round(mlo, 4), "mae_hi": round(mhi, 4),
                         "rmse": round(math.sqrt(mean_squared_error(y, oof)), 4),
                         "fold_r2_min": round(fm.r2.min(), 4), "fold_r2_max": round(fm.r2.max(), 4)})
        OOF[(name, scheme)] = oof
    print(f"  done: {name}")

T11 = pd.DataFrame(lad_rows)
print("\nMODEL LADDER")
print(T11.to_string(index=False))
save_table(T11, "T11_model_ladder",
           "Model ladder performance under random and spatial block cross-validation, "
           "with bootstrap 95% confidence intervals.")

for _, r in T11.iterrows():
    key = r.model.split()[0] + "_" + ("rand" if "random" in r.scheme else "block")
    rec(f"ml::{key}_r2", r.r2)
    rec(f"ml::{key}_r2_ci", [r.r2_lo, r.r2_hi])
    rec(f"ml::{key}_mae", r.mae)


training set n = 459
  demographic features : 4
  open-data features   : 34
spatial blocks: 10, sizes [ 57  66 146   8  27  28  30  53  23  21]
loaded CV cache with 34 entries (set USE_CACHE = False to recompute from scratch)


  done: M0  demographics only (OLS)


  done: M1  + open data (OLS)


  done: M1r + open data (ridge)


  done: M2  + open data (gradient boosting)


  done: M2r + open data (random forest)

MODEL LADDER
                              model        scheme  n_features      r2   r2_lo   r2_hi    mae  mae_lo  mae_hi   rmse  fold_r2_min  fold_r2_max
        M0  demographics only (OLS) random 5-fold           4  0.1047  0.0400  0.1609 0.5785  0.5383  0.6157 0.7250      -0.0060       0.1978
        M0  demographics only (OLS) spatial block           4  0.0799  0.0138  0.1391 0.5876  0.5468  0.6250 0.7349      -0.1708       0.0853
              M1  + open data (OLS) random 5-fold          38  0.0961  0.0045  0.1771 0.5836  0.5435  0.6212 0.7285       0.0097       0.1612
              M1  + open data (OLS) spatial block          38 -0.1268 -0.2483 -0.0197 0.6522  0.6074  0.6958 0.8133      -0.4484       0.1041
            M1r + open data (ridge) random 5-fold          38  0.1383  0.0751  0.1935 0.5782  0.5410  0.6128 0.7112       0.0712       0.1668
            M1r + open data (ridge) spatial block          38 -0.0006 -0.0934  0.0816 0.6154  

### 5.2 — The rung the outline did not anticipate

The ladder above shows that 34 open-data features add nothing to four demographic
variables once spatial leakage is controlled. That is a result about *open data*, not
about the archive. The decisive comparison is therefore a fourth rung: what does the
**rest of the survey archive** — the non-burden Likert items already sitting unused in
the same file — contribute?

If M3 beats M1 and M2, the paper's central claim stops being an argument and becomes a
measurement: the scarce, expensive, institutionally hard-to-sustain layer is the one that
carries the signal, and the abundant free layer is not a substitute for it.


In [13]:
F_SURVEY = [c for c in ["i_drain", "i_facil", "i_road",
                        "p_security", "p_change", "p_advantage", "p_green"]
            if c in D.columns and D[c].notna().mean() > 0.90]
print(f"non-target survey items available as features: {F_SURVEY}")
rec("ml_n_survey_features", len(F_SURVEY))

LADDER2 = [
    ("M3  demographics + survey archive (OLS)",       F_DEMOG + F_SURVEY, "ols", False),
    ("M4  demographics + survey + open data (ridge)", F_DEMOG + F_SURVEY + F_OPEN, "ridge", False),
    ("M5  demographics + survey + open data (GBM)",   F_DEMOG + F_SURVEY + F_OPEN, "gbm", RUN_NESTED_CV),
]
rows2 = []
for name, feats, kind, tune in LADDER2:
    X = D[feats]
    for scheme, groups in [("random 5-fold", None), ("spatial block", D.block.values)]:
        oof, fm = cached(f"cv|{name}|{scheme}",
                         lambda X=X, kind=kind, groups=groups, tune=tune:
                             nested_cv(X, y, kind, groups=groups, tune=tune))
        lo, hi = boot_ci(y, oof, "r2"); mlo, mhi = boot_ci(y, oof, "mae")
        rows2.append({"model": name, "scheme": scheme, "n_features": len(feats),
                      "r2": round(r2_score(y, oof), 4), "r2_lo": round(lo, 4), "r2_hi": round(hi, 4),
                      "mae": round(mean_absolute_error(y, oof), 4),
                      "mae_lo": round(mlo, 4), "mae_hi": round(mhi, 4),
                      "rmse": round(math.sqrt(mean_squared_error(y, oof)), 4),
                      "fold_r2_min": round(fm.r2.min(), 4), "fold_r2_max": round(fm.r2.max(), 4)})
        OOF[(name, scheme)] = oof
    print(f"  done: {name}")

T11 = pd.concat([T11, pd.DataFrame(rows2)], ignore_index=True)
save_table(T11, "T11_model_ladder",
           "Model ladder performance under random and spatial block cross-validation, "
           "with bootstrap 95% confidence intervals.")
print("\nFULL LADDER")
print(T11.to_string(index=False))
for _, r in T11.iterrows():
    key = r.model.split()[0] + "_" + ("rand" if "random" in r.scheme else "block")
    rec(f"ml::{key}_r2", r.r2); rec(f"ml::{key}_r2_ci", [r.r2_lo, r.r2_hi]); rec(f"ml::{key}_mae", r.mae)


non-target survey items available as features: ['i_drain', 'i_facil', 'i_road', 'p_security', 'p_change', 'p_advantage', 'p_green']


  done: M3  demographics + survey archive (OLS)


  done: M4  demographics + survey + open data (ridge)


  done: M5  demographics + survey + open data (GBM)
  -> Tables/T11_model_ladder.csv  (16 rows)

FULL LADDER
                                        model        scheme  n_features      r2   r2_lo   r2_hi    mae  mae_lo  mae_hi   rmse  fold_r2_min  fold_r2_max
                  M0  demographics only (OLS) random 5-fold           4  0.1047  0.0400  0.1609 0.5785  0.5383  0.6157 0.7250      -0.0060       0.1978
                  M0  demographics only (OLS) spatial block           4  0.0799  0.0138  0.1391 0.5876  0.5468  0.6250 0.7349      -0.1708       0.0853
                        M1  + open data (OLS) random 5-fold          38  0.0961  0.0045  0.1771 0.5836  0.5435  0.6212 0.7285       0.0097       0.1612
                        M1  + open data (OLS) spatial block          38 -0.1268 -0.2483 -0.0197 0.6522  0.6074  0.6958 0.8133      -0.4484       0.1041
                      M1r + open data (ridge) random 5-fold          38  0.1383  0.0751  0.1935 0.5782  0.5410  0.6128 0.7112      

### 5.3 — Paired increments with confidence intervals

Each increment is tested by paired bootstrap on the out-of-fold predictions, so the two
models being compared are always evaluated on exactly the same resampled observations.


In [14]:
def paired_delta(name_a, name_b, scheme, n=N_BOOT, seed=RNG):
    """Bootstrap CI for R2(b) - R2(a) on shared out-of-fold predictions."""
    pa, pb = OOF[(name_a, scheme)], OOF[(name_b, scheme)]
    rg = np.random.default_rng(seed)
    idx = np.arange(len(y)); ds = []
    for _ in range(n):
        b = rg.choice(idx, size=len(idx), replace=True)
        ds.append(r2_score(y[b], pb[b]) - r2_score(y[b], pa[b]))
    ds = np.array(ds)
    obs = r2_score(y, pb) - r2_score(y, pa)
    p = 2 * min((ds <= 0).mean(), (ds >= 0).mean())
    return obs, float(np.percentile(ds, 2.5)), float(np.percentile(ds, 97.5)), float(max(p, 1 / n))


M0 = "M0  demographics only (OLS)"
M1r = "M1r + open data (ridge)"
M2g = "M2  + open data (gradient boosting)"
M3 = "M3  demographics + survey archive (OLS)"
M5 = "M5  demographics + survey + open data (GBM)"

COMPARISONS = [
    ("open data added to demographics (linear)",      M0,  M1r),
    ("open data added to demographics (nonlinear)",   M0,  M2g),
    ("survey archive added to demographics",          M0,  M3),
    ("open data added on top of the survey archive",  M3,  M5),
    ("nonlinearity added on top of linear open data", M1r, M2g),
]
inc_rows = []
for lab, a, b in COMPARISONS:
    for scheme in ["random 5-fold", "spatial block"]:
        obs, lo, hi, p = paired_delta(a, b, scheme)
        inc_rows.append({"increment": lab, "scheme": scheme,
                         "delta_r2": round(obs, 4), "ci_low": round(lo, 4),
                         "ci_high": round(hi, 4), "p_boot": round(p, 4),
                         "significant": "yes" if (lo > 0 or hi < 0) else "no"})
T12 = pd.DataFrame(inc_rows)
print("PAIRED INCREMENTS IN R^2 (bootstrap 95% CI on shared out-of-fold predictions)")
print(T12.to_string(index=False))
save_table(T12, "T12_paired_increments",
           "Paired bootstrap increments in out-of-fold R-squared between successive model rungs.")
for _, r in T12.iterrows():
    tag = ("rand" if "random" in r.scheme else "block")
    rec(f"delta::{r.increment} [{tag}]", {"d": r.delta_r2, "ci": [r.ci_low, r.ci_high],
                                          "sig": r.significant})

# ---- the optimism gap: random CV minus spatial block CV, per model
opt = (T11.pivot_table(index="model", columns="scheme", values="r2")
       .assign(optimism=lambda t: t["random 5-fold"] - t["spatial block"])
       .sort_values("optimism", ascending=False).round(4).reset_index())
print("\nSPATIAL OPTIMISM (random-CV R^2 minus block-CV R^2)")
print(opt.to_string(index=False))
save_table(opt, "T13_spatial_optimism",
           "Spatial optimism: the inflation of R-squared under random cross-validation.")
rec("optimism_max", float(opt.optimism.max()))
rec("optimism_max_model", opt.iloc[0].model)
rec("optimism_min", float(opt.optimism.min()))

# ---- the headline
best_block = T11[T11.scheme == "spatial block"].sort_values("r2", ascending=False).iloc[0]
print(f"\nBEST MODEL UNDER SPATIAL BLOCK CV: {best_block.model}")
print(f"  R2 = {best_block.r2:+.4f} [{best_block.r2_lo:+.4f}, {best_block.r2_hi:+.4f}], "
      f"MAE = {best_block.mae:.3f} on a 1-5 scale")
rec("best_block_model", best_block.model)
rec("best_block_r2", float(best_block.r2))
rec("best_block_r2_ci", [float(best_block.r2_lo), float(best_block.r2_hi)])
rec("best_block_mae", float(best_block.mae))
rec("best_block_r2_as_pct", round(100 * float(best_block.r2), 1))
BEST = best_block.model


PAIRED INCREMENTS IN R^2 (bootstrap 95% CI on shared out-of-fold predictions)
                                    increment        scheme  delta_r2  ci_low  ci_high  p_boot significant
     open data added to demographics (linear) random 5-fold    0.0336 -0.0137   0.0821  0.1750          no
     open data added to demographics (linear) spatial block   -0.0805 -0.1699   0.0044  0.0600          no
  open data added to demographics (nonlinear) random 5-fold    0.0048 -0.0555   0.0665  0.8560          no
  open data added to demographics (nonlinear) spatial block   -0.0092 -0.0866   0.0654  0.8160          no
         survey archive added to demographics random 5-fold    0.2026  0.1413   0.2647  0.0005         yes
         survey archive added to demographics spatial block    0.2125  0.1502   0.2781  0.0005         yes
 open data added on top of the survey archive random 5-fold   -0.0106 -0.0614   0.0441  0.7030          no
 open data added on top of the survey archive spatial block   -0.0

## Step 6 — Interpretation: SHAP on the full-feature model

SHAP is computed on the model that sees *everything* (M5: demographics + survey archive +
open data, gradient boosting), because the interpretive question is not "which model wins"
but "when a model is free to use any of the 45 available features, which ones does it
actually use?" That is the question a decision-maker asks of a decision engine.


In [15]:
import shap

F_FULL = F_DEMOG + F_SURVEY + F_OPEN
Xf = D[F_FULL]
gbm_full = make_model("gbm")
gbm_full.set_params(m__max_depth=3, m__learning_rate=0.06, m__max_leaf_nodes=15,
                    m__min_samples_leaf=20, m__l2_regularization=1.0)
gbm_full.fit(Xf, y)

Ximp = pd.DataFrame(gbm_full.named_steps["imp"].transform(Xf), columns=F_FULL)
if RUN_SHAP:
    expl = shap.TreeExplainer(gbm_full.named_steps["m"])
    SV = expl.shap_values(Ximp)
else:
    SV = np.zeros((len(Ximp), len(F_FULL)))

FEATURE_BLOCK = ({f: "Demographics" for f in F_DEMOG} |
                 {f: "Survey archive" for f in F_SURVEY} |
                 {f: "Open data" for f in F_OPEN})
BLOCK_COLOR = {"Demographics": "#c26a3d", "Survey archive": "#1b4965", "Open data": "#6b8f71"}

mean_abs = np.abs(SV).mean(axis=0)
SHAPIMP = (pd.DataFrame({"feature": F_FULL, "mean_abs_shap": mean_abs,
                         "block": [FEATURE_BLOCK[f] for f in F_FULL]})
           .sort_values("mean_abs_shap", ascending=False).reset_index(drop=True))
SHAPIMP["share_pct"] = (100 * SHAPIMP.mean_abs_shap / SHAPIMP.mean_abs_shap.sum()).round(2)
SHAPIMP["rank"] = np.arange(1, len(SHAPIMP) + 1)
SHAPIMP["mean_abs_shap"] = SHAPIMP.mean_abs_shap.round(5)
print("Top 15 features by mean |SHAP|:")
print(SHAPIMP.head(15).to_string(index=False))
save_table(SHAPIMP, "T14_shap_importance",
           "Global SHAP feature importance for the full-feature gradient boosting model.")

blk = SHAPIMP.groupby("block").agg(n_features=("feature", "size"),
                                   total_share=("share_pct", "sum")).reset_index()
blk["share_per_feature"] = (blk.total_share / blk.n_features).round(2)
blk = blk.sort_values("total_share", ascending=False)
print("\nAttribution by feature block:")
print(blk.to_string(index=False))
save_table(blk, "T15_shap_by_block",
           "SHAP attribution aggregated by feature block.")
for _, r in blk.iterrows():
    rec(f"shap_block_share::{r.block}", float(r.total_share))
    rec(f"shap_block_per_feature::{r.block}", float(r.share_per_feature))
rec("shap_top_feature", SHAPIMP.iloc[0].feature)
rec("shap_top_feature_share", float(SHAPIMP.iloc[0].share_pct))
rec("shap_top2_feature_share", float(SHAPIMP.iloc[1].share_pct))
rec("shap_top5", list(SHAPIMP.head(5).feature))
top15 = SHAPIMP.head(15)
rec("shap_top15_open_data_count", int((top15.block == "Open data").sum()))
rec("shap_top15_survey_count", int((top15.block == "Survey archive").sum()))
print(f"\nof the 15 most important features, {int((top15.block=='Open data').sum())} come from "
      f"open data and {int((top15.block=='Survey archive').sum())} from the survey archive "
      f"({len(F_OPEN)} of the 45 candidate features are open-data features)")


Top 15 features by mean |SHAP|:
    feature  mean_abs_shap          block  share_pct  rank
    i_drain        0.19231 Survey archive      17.27     1
     i_road        0.13632 Survey archive      12.24     2
        edu        0.08552   Demographics       7.68     3
p_advantage        0.08124 Survey archive       7.29     4
  d_water_m        0.07187      Open data       6.45     5
     d_ndbi        0.04216      Open data       3.79     6
        age        0.03355   Demographics       3.01     7
 p_security        0.03288 Survey archive       2.95     8
       d_vh        0.03160      Open data       2.84     9
     income        0.02974   Demographics       2.67    10
elev_m_r300        0.02524      Open data       2.27    11
      vh_p1        0.02486      Open data       2.23    12
     d_ndvi        0.02356      Open data       2.12    13
  slope_deg        0.02273      Open data       2.04    14
     elev_m        0.02115      Open data       1.90    15
  -> Tables/T14_shap_imp

## Step 7 — Spatial accountability: where the model fails

A decision engine that reports only predictions is not accountable. This step maps the
residual and tests whether spatial structure remains after the model has done its work.


In [16]:
BEST_SCHEME = "spatial block"
pred_best = OOF[(BEST, BEST_SCHEME)]
D["pred"] = pred_best
D["residual"] = y - pred_best
D["abs_residual"] = np.abs(D["residual"])

mi_resid = Moran(D["residual"].values, W, permutations=999)
mi_raw = Moran(y, W, permutations=999)
print(f"Moran's I of the target      = {mi_raw.I:+.4f} (p = {mi_raw.p_sim:.4f})")
print(f"Moran's I of the residual    = {mi_resid.I:+.4f} (p = {mi_resid.p_sim:.4f})")
print(f"proportion of spatial structure removed = "
      f"{100*(1 - mi_resid.I/mi_raw.I):.1f}%")
rec("moran_residual", round(float(mi_resid.I), 4))
rec("moran_residual_p", float(mi_resid.p_sim))
rec("moran_structure_removed_pct", round(100 * (1 - mi_resid.I / mi_raw.I), 1))

# LISA on the absolute residual: where is the model systematically unreliable?
from esda.moran import Moran_Local
lisa = Moran_Local(D["abs_residual"].values, W, permutations=999, seed=RNG)
q_sig = stats.false_discovery_control(lisa.p_sim, method="bh") < 0.05
D["resid_cluster"] = np.where(q_sig & (lisa.q == 1), "high-error cluster",
                      np.where(q_sig & (lisa.q == 3), "low-error cluster", "not significant"))
print("\nLISA on absolute residual (BH-corrected q < .05):")
print(D.resid_cluster.value_counts().to_string())
rec("n_high_error_cluster", int((D.resid_cluster == "high-error cluster").sum()))
rec("n_low_error_cluster", int((D.resid_cluster == "low-error cluster").sum()))

# characterise the high-error cluster against the rest
hi = D.resid_cluster == "high-error cluster"
if hi.sum() >= 5:
    prof_rows = []
    for f in ["burden_mean", "elev_m", "hand_m", "d_water_m", "built_frac", "pop_dens",
              "ndvi", "lst_c", "road_density", "d_metro_m", "edu", "income"]:
        if f not in D.columns:
            continue
        a_, b_ = D.loc[hi, f].dropna(), D.loc[~hi, f].dropna()
        if len(a_) < 3:
            continue
        u_, p_ = stats.mannwhitneyu(a_, b_, alternative="two-sided")
        prof_rows.append({"feature": f, "high_error_median": round(float(a_.median()), 3),
                          "rest_median": round(float(b_.median()), 3),
                          "p": round(float(p_), 4)})
    PROF = pd.DataFrame(prof_rows)
    PROF["p_bh"] = stats.false_discovery_control(PROF.p.values, method="bh").round(4)
    print("\nprofile of the high-error cluster:")
    print(PROF.to_string(index=False))
    save_table(PROF, "T16_high_error_profile",
               "Characteristics of the spatial cluster where the model errs most.")
    rec("high_error_sig_features", list(PROF[PROF.p_bh < 0.05].feature))

D[["lat", "lon", "x", "y", "block", TARGET, "pred", "residual",
   "abs_residual", "resid_cluster"]].to_csv(RES / "residuals.csv", index=False)
print("\n  -> Results/residuals.csv")


Moran's I of the target      = +0.1402 (p = 0.0010)
Moran's I of the residual    = +0.0820 (p = 0.0010)
proportion of spatial structure removed = 41.5%



LISA on absolute residual (BH-corrected q < .05):
resid_cluster
not significant      449
low-error cluster     10

  -> Results/residuals.csv


## Step 8 — Transfer: does the pipeline move to a different stakeholder type?

The model is trained on residents (n = 459) and applied unchanged to the SME instrument
(n = 80): a different questionnaire, a different respondent type, the same corridor and
the same feature layer. This is the pipeline's portability test.


In [17]:
S = MT[MT.group == "SMEs"].copy().reset_index(drop=True)
S = S.dropna(subset=[TARGET, "x", "y"]).reset_index(drop=True)
print(f"transfer target: SMEs n = {len(S)}")

# The SME instrument does not carry the demographic block, so transfer is tested on the
# feature sets both instruments share.
F_SHARED_SURVEY = [f for f in F_SURVEY if S[f].notna().mean() > 0.80]
F_SHARED_OPEN = [f for f in F_OPEN if S[f].notna().mean() > 0.80]
print(f"shared survey features   : {len(F_SHARED_SURVEY)} {F_SHARED_SURVEY}")
print(f"shared open-data features: {len(F_SHARED_OPEN)}")
rec("transfer_n_smes", int(len(S)))
rec("transfer_shared_survey_features", F_SHARED_SURVEY)

TRANSFER_SETS = [
    ("open data only",            F_SHARED_OPEN),
    ("survey archive only",       F_SHARED_SURVEY),
    ("survey archive + open data", F_SHARED_SURVEY + F_SHARED_OPEN),
]
tr_rows = []
for label, feats in TRANSFER_SETS:
    if not feats:
        continue
    for kind in ["ols", "gbm"]:
        est = make_model(kind).fit(D[feats], y)
        # in-domain honesty check: block-CV on residents with the same feature set
        oof_in, _ = cached(f"tr|{label}|{kind}",
                           lambda feats=feats, kind=kind:
                               nested_cv(D[feats], y, kind, groups=D.block.values))
        r2_in = r2_score(y, oof_in)
        p_out = est.predict(S[feats])
        r2_out = r2_score(S[TARGET].values, p_out)
        lo, hi = boot_ci(S[TARGET].values, p_out, "r2")
        rho, p_rho = stats.spearmanr(S[TARGET].values, p_out)
        tr_rows.append({"feature_set": label, "model": kind.upper(),
                        "n_features": len(feats),
                        "r2_residents_blockcv": round(r2_in, 4),
                        "r2_smes_transfer": round(r2_out, 4),
                        "r2_smes_ci": f"[{lo:.3f}, {hi:.3f}]",
                        "mae_smes": round(mean_absolute_error(S[TARGET], p_out), 4),
                        "spearman_rho": round(float(rho), 3),
                        "spearman_p": round(float(p_rho), 4)})
T17 = pd.DataFrame(tr_rows)
print("\nTRANSFER: residents -> SMEs")
print(T17.to_string(index=False))
save_table(T17, "T17_transfer_residents_to_smes",
           "Transfer of the trained model from residents to the SME instrument.")
best_tr = T17.sort_values("spearman_rho", ascending=False).iloc[0]
print(f"\nbest transfer by rank correlation: {best_tr.feature_set} ({best_tr.model}), "
      f"rho = {best_tr.spearman_rho:+.3f}, p = {best_tr.spearman_p:.4f}")
rec("transfer_best_set", best_tr.feature_set)
rec("transfer_best_r2", float(best_tr.r2_smes_transfer))
rec("transfer_best_model", best_tr.model)
rec("transfer_opendata_rho", float(T17[T17.feature_set=="open data only"].spearman_rho.max()))
rec("transfer_opendata_rho_p", float(T17[T17.feature_set=="open data only"].spearman_p.min()))
rec("transfer_best_rho", float(best_tr.spearman_rho))
rec("transfer_best_rho_p", float(best_tr.spearman_p))

# level shift: SMEs report systematically lower burden (Paper 1's self-selection result)
tstat, tp = stats.mannwhitneyu(D[TARGET], S[TARGET], alternative="two-sided")
print(f"\nburden level: residents {D[TARGET].mean():.3f} vs SMEs {S[TARGET].mean():.3f} "
      f"(Mann-Whitney p = {tp:.2e})")
print("  a level shift of this size caps transfer R^2 regardless of how well the model ranks;")
print("  Spearman rho is therefore the informative transfer statistic, and is reported as such.")
rec("burden_residents_mean", round(float(D[TARGET].mean()), 3))
rec("burden_smes_mean", round(float(S[TARGET].mean()), 3))
rec("burden_level_p", float(tp))


transfer target: SMEs n = 80
shared survey features   : 7 ['i_drain', 'i_facil', 'i_road', 'p_security', 'p_change', 'p_advantage', 'p_green']
shared open-data features: 34



TRANSFER: residents -> SMEs
               feature_set model  n_features  r2_residents_blockcv  r2_smes_transfer       r2_smes_ci  mae_smes  spearman_rho  spearman_p
            open data only   OLS          34               -0.3514           -0.2947  [-0.840, 0.013]    0.6557         0.212      0.0589
            open data only   GBM          34               -0.2395           -0.3475 [-0.903, -0.013]    0.6788         0.189      0.0933
       survey archive only   OLS           7                0.2533           -0.2777  [-0.859, 0.044]    0.6472         0.328      0.0029
       survey archive only   GBM           7                0.2244           -0.3965  [-1.073, 0.008]    0.6630         0.227      0.0426
survey archive + open data   OLS          41               -0.0350           -0.2673  [-0.874, 0.094]    0.6457         0.349      0.0015
survey archive + open data   GBM          41                0.1004           -0.2075  [-0.708, 0.119]    0.6231         0.269      0.0159
  -> 

2.1567953770681242e-06

## Step 9 — Three knowledges: which burdens can the data see?

Paper 2 established an Expert–Authority Alignment Index over the six urban issues.
This step adds the third axis the outline calls for, but in a sharper form than a raw
SHAP-versus-expert rank correlation: for each of the six burdens separately, how much of
it can the pipeline predict at all under spatial block cross-validation?

A burden the model cannot predict is a burden the available data cannot see. That is a
statement about the *instrument stock of the city*, not about the residents who report it.


In [18]:
ISSUE_ITEM = {"air": "b_air", "flood": "b_flood", "noise": "b_noise",
              "water": "b_water", "congest": "b_congest", "accident": "b_accident"}
leg_rows = []
FEATS_LEG = F_DEMOG + [f for f in F_SURVEY] + F_OPEN
for iss, item in ISSUE_ITEM.items():
    yi = D[item].values
    ok = np.isfinite(yi)
    feats = [f for f in FEATS_LEG]
    oof_o, _ = cached(f"leg|{item}|open", lambda: nested_cv(
        D.loc[ok, F_DEMOG + F_OPEN], yi[ok], "gbm", groups=D.loc[ok, "block"].values))
    oof_f, _ = cached(f"leg|{item}|full", lambda: nested_cv(
        D.loc[ok, feats], yi[ok], "gbm", groups=D.loc[ok, "block"].values))
    mi = Moran(yi[ok], KNN.from_array(D.loc[ok, ["x", "y"]].values, k=8), permutations=999)
    leg_rows.append({
        "issue": ISSUE_EN[iss], "item": item, "n": int(ok.sum()),
        "resident_mean": round(float(np.nanmean(yi)), 3),
        "morans_I": round(float(mi.I), 4), "morans_p": round(float(mi.p_sim), 4),
        "r2_open_data_only": round(r2_score(yi[ok], oof_o), 4),
        "r2_full": round(r2_score(yi[ok], oof_f), 4)})
LEG = pd.DataFrame(leg_rows)

# expert and authority rankings of the same six issues
exp_sev = {}
for iss, q in EXP_ISSUE_Q.items():
    v = EXP[q].map(SCALE_SEVERITY).dropna()
    exp_sev[ISSUE_EN[iss]] = round(float(v.mean()), 3)
LEG["expert_severity"] = LEG.issue.map(exp_sev)

AUTH_ISSUE = {"Ùn tắc giao thông": "Traffic congestion", "Tai nạn giao thông": "Traffic accidents",
              "Ô nhiễm không khí": "Air pollution", "Ô nhiễm âm thanh": "Noise pollution",
              "Ô nhiễm nguồn nước": "Water pollution", "Ngập lụt": "Flooding"}
auth_rows = split_multi(AUT["Q2.2"])
auth_flat = [AUTH_ISSUE.get(b.strip()) for r in auth_rows for b in r]
auth_ct = pd.Series([a for a in auth_flat if a]).value_counts()
LEG["authority_agenda_pct"] = LEG.issue.map(lambda s: round(100 * auth_ct.get(s, 0) / len(auth_rows), 1))

for c, asc in [("resident_mean", False), ("expert_severity", False),
               ("authority_agenda_pct", False), ("r2_full", False)]:
    LEG[c.replace("_mean", "").replace("_pct", "") + "_rank"] = LEG[c].rank(ascending=asc).astype(int)
LEG = LEG.sort_values("resident_mean", ascending=False)
print("SIX BURDENS ACROSS FOUR SOURCES OF KNOWLEDGE")
print(LEG.to_string(index=False))
save_table(LEG, "T18_three_knowledges",
           "The six urban burdens ranked by residents, experts, the institutional agenda, "
           "and by how well the available data can predict them.")

pairs_rho = []
COLS = {"residents": "resident_mean", "experts": "expert_severity",
        "authority agenda": "authority_agenda_pct", "data legibility": "r2_full"}
for (na, ca), (nb, cb) in itertools.combinations(COLS.items(), 2):
    rho, p = stats.spearmanr(LEG[ca], LEG[cb])
    pairs_rho.append({"source_a": na, "source_b": nb,
                      "spearman_rho": round(float(rho), 3), "p": round(float(p), 4)})
RHO = pd.DataFrame(pairs_rho)
print("\nagreement between sources (Spearman, 6 issues - n is small, treat as descriptive):")
print(RHO.to_string(index=False))
save_table(RHO, "T19_source_agreement",
           "Rank agreement between residents, experts, the institutional agenda and data legibility.")
rec("legibility_best", LEG.sort_values("r2_full", ascending=False).iloc[0].issue)
rec("legibility_worst", LEG.sort_values("r2_full").iloc[0].issue)
rec("legibility_range", [round(float(LEG.r2_full.min()), 4), round(float(LEG.r2_full.max()), 4)])
for _, r in RHO.iterrows():
    rec(f"rho::{r.source_a} vs {r.source_b}", r.spearman_rho)


SIX BURDENS ACROSS FOUR SOURCES OF KNOWLEDGE
             issue       item   n  resident_mean  morans_I  morans_p  r2_open_data_only  r2_full  expert_severity  authority_agenda_pct  resident_rank  expert_severity_rank  authority_agenda_rank  r2_full_rank
     Air pollution      b_air 459          2.651    0.0604     0.005            -0.1913  -0.0646            3.290                  65.2              1                     4                      3             6
   Noise pollution    b_noise 459          2.547    0.0678     0.002            -0.1768  -0.0314            3.400                  17.4              2                     1                      6             5
Traffic congestion  b_congest 459          2.514    0.0784     0.001            -0.0521   0.0556            3.303                  95.7              3                     3                      1             3
          Flooding    b_flood 459          2.484    0.1834     0.001            -0.2181   0.1416            3.000  

### Figures 3, 4, 5 — model ladder, SHAP, spatial accountability


In [19]:
# ---------------------------------------------------------------- Figure 3
fig = plt.figure(figsize=(13.2, 5.2))
gs = fig.add_gridspec(1, 2, width_ratios=[1.45, 1.0], wspace=0.30)

ax = fig.add_subplot(gs[0, 0])
order = ["M0  demographics only (OLS)", "M1  + open data (OLS)", "M1r + open data (ridge)",
         "M2  + open data (gradient boosting)", "M2r + open data (random forest)",
         "M3  demographics + survey archive (OLS)",
         "M4  demographics + survey + open data (ridge)",
         "M5  demographics + survey + open data (GBM)"]
SHORT = {order[0]: "M0  demographics only",
         order[1]: "M1  + open data (OLS)",
         order[2]: "M1r + open data (ridge)",
         order[3]: "M2  + open data (GBM)",
         order[4]: "M2r + open data (RF)",
         order[5]: "M3  + survey archive",
         order[6]: "M4  + survey + open (ridge)",
         order[7]: "M5  + survey + open (GBM)"}
yy = np.arange(len(order))[::-1]
h = 0.34
for scheme, off, col, mk in [("random 5-fold", +h/2, MUTED, "o"),
                             ("spatial block", -h/2, ACCENT, "s")]:
    sub = T11[T11.scheme == scheme].set_index("model").loc[order]
    ax.errorbar(sub.r2, yy + off,
                xerr=[sub.r2 - sub.r2_lo, sub.r2_hi - sub.r2],
                fmt=mk, color=col, ms=6, elinewidth=1.6, capsize=3.5,
                label=f"{scheme} CV", zorder=4)
ax.axvline(0, color=INK, lw=1.0, zorder=2)
ax.set_yticks(yy); ax.set_yticklabels([SHORT[m] for m in order], fontsize=8.5)
ax.set_xlabel("out-of-fold $R^2$ (bars: bootstrap 95% CI)")
ax.set_title("a  Model ladder: what actually predicts reported burden", loc="left")
ax.legend(fontsize=8.5, loc="upper left")
ax.axhspan(-0.55, 2.55, color=ACCENT, alpha=0.06, zorder=0)
ax.annotate("models that use the\nsurvey archive", xy=(0.03, 0.10), xycoords="axes fraction",
            ha="left", va="center", fontsize=8.5, color=ACCENT, style="italic")
ax.set_ylim(-0.8, len(order) - 0.2)

ax = fig.add_subplot(gs[0, 1])
dinc = T12[T12.scheme == "spatial block"].reset_index(drop=True)
yy2 = np.arange(len(dinc))[::-1]
colr = [ACCENT if s == "yes" and d > 0 else ("#b23a48" if s == "yes" else MUTED)
        for s, d in zip(dinc.significant, dinc.delta_r2)]
for i, (_, r) in enumerate(dinc.iterrows()):
    ax.errorbar([r.delta_r2], [yy2[i]],
                xerr=[[r.delta_r2 - r.ci_low], [r.ci_high - r.delta_r2]],
                fmt="none", ecolor=colr[i], elinewidth=1.8, capsize=3.5, zorder=3)
ax.scatter(dinc.delta_r2, yy2, color=colr, s=46, zorder=4)
ax.axvline(0, color=INK, lw=1.0, zorder=2)
ax.set_yticks(yy2); ax.set_yticklabels([wrap(s, 30) for s in dinc.increment], fontsize=8)
ax.set_xlabel("$\\Delta R^2$ under spatial block CV")
ax.set_title("b  Paired increments\n(blue = significant gain, red = significant loss)", loc="left")
fig.savefig(FIG / "Figure3_model_ladder.png")
plt.close(fig)
print("  -> Figures/Figure3_model_ladder.png")

# ---------------------------------------------------------------- Figure 4  SHAP
FEATURE_LABEL = {
    "i_drain": "Drainage adequacy (survey)", "i_road": "Road adequacy (survey)",
    "i_facil": "Public facilities (survey)", "p_security": "Perceived security (survey)",
    "p_change": "Area is changing well (survey)", "p_advantage": "Area advantage (survey)",
    "p_green": "Green space access (survey)",
    "edu": "Education", "income": "Household income", "age": "Age", "female": "Female",
    "elev_m": "Elevation", "slope_deg": "Slope", "elev_m_r300": "Elevation, 300 m mean",
    "hand_m": "Height above drainage", "d_water_m": "Distance to water",
    "flood_freq": "JRC water occurrence", "water_recur": "JRC water recurrence",
    "water_max": "JRC water extent", "flood_freq_r300": "JRC occurrence, 300 m",
    "flood_sar": "SAR open-water flag", "vh_p1": "SAR VH backscatter (2022–23)",
    "vh_p2": "SAR VH backscatter (2025–26)", "d_vh": "Δ SAR VH backscatter",
    "flood_sar_r300": "SAR flag, 300 m", "ndvi": "NDVI", "ndbi": "NDBI",
    "ndwi": "NDWI", "mndwi": "MNDWI", "ndvi_r300": "NDVI, 300 m mean",
    "ndbi_r300": "NDBI, 300 m mean", "ndvi_p2": "NDVI (2025–26)", "ndbi_p2": "NDBI (2025–26)",
    "d_ndvi": "Δ NDVI", "d_ndbi": "Δ NDBI", "d_ndvi_r300": "Δ NDVI, 300 m",
    "lst_c": "Land surface temperature", "lst_p2": "LST (2025–26)", "d_lst": "Δ LST",
    "pop_dens": "Population density (GHSL)", "pop_2025": "Population 2025 (GHSL)",
    "built_m2": "Built surface (GHSL)", "built_2025": "Built surface 2025 (GHSL)",
    "d_pop": "Δ population", "d_built": "Δ built surface", "built_frac": "Built fraction",
    "road_density": "Road density (OSM)", "d_metro_m": "Distance to metro station",
}
flab = lambda f: FEATURE_LABEL.get(f, f)
SHAPIMP["label"] = SHAPIMP.feature.map(flab)
save_table(SHAPIMP, "T14_shap_importance",
           "Global SHAP feature importance for the full-feature gradient boosting model.")

fig = plt.figure(figsize=(13.6, 5.6))
gs = fig.add_gridspec(1, 3, width_ratios=[1.15, 1.35, 0.75], wspace=0.40)

ax = fig.add_subplot(gs[0, 0])
t15 = SHAPIMP.head(15).iloc[::-1]
yy = np.arange(len(t15))
ax.barh(yy, t15.mean_abs_shap, color=[BLOCK_COLOR[b] for b in t15.block], height=0.68, zorder=3)
ax.set_yticks(yy); ax.set_yticklabels(t15.label, fontsize=8)
ax.set_xlabel("mean |SHAP| (burden units, 1–5 scale)")
ax.set_title("a  Global feature importance\n(full 45-feature GBM)", loc="left")
handles = [Patch(facecolor=BLOCK_COLOR[b], label=b) for b in
           ["Survey archive", "Demographics", "Open data"]]
ax.legend(handles=handles, fontsize=8, loc="lower right")

ax = fig.add_subplot(gs[0, 1])
topf = list(SHAPIMP.head(12).feature)
sv_idx = {f: i for i, f in enumerate(F_FULL)}
for j, f in enumerate(topf[::-1]):
    sv = SV[:, sv_idx[f]]
    xv = Ximp[f].values
    xn = (xv - np.nanmin(xv)) / (np.ptp(xv) + 1e-12)
    jit = (np.random.default_rng(RNG + j).random(len(sv)) - 0.5) * 0.34
    ax.scatter(sv, np.full(len(sv), j) + jit, c=xn, cmap="coolwarm", s=4.5,
               alpha=0.55, linewidths=0, zorder=3, vmin=0, vmax=1)
ax.axvline(0, color=INK, lw=0.9, zorder=2)
ax.set_yticks(range(len(topf))); ax.set_yticklabels([flab(f) for f in topf[::-1]], fontsize=8)
ax.set_xlabel("SHAP value (effect on predicted burden)")
ax.set_title("b  Beeswarm — direction and spread\n(red = high feature value)", loc="left")
sm = plt.cm.ScalarMappable(cmap="coolwarm", norm=plt.Normalize(0, 1))
cb = fig.colorbar(sm, ax=ax, pad=0.015, fraction=0.030)
cb.set_ticks([0, 1]); cb.set_ticklabels(["low", "high"]); cb.ax.tick_params(labelsize=7.5)
cb.outline.set_visible(False)

ax = fig.add_subplot(gs[0, 2])
b = blk.set_index("block")
labels_b = [l for l in ["Survey archive", "Demographics", "Open data"] if l in b.index]
yy3 = np.arange(len(labels_b))[::-1]
ax.barh(yy3, [b.loc[l, "share_per_feature"] for l in labels_b],
        color=[BLOCK_COLOR[l] for l in labels_b], height=0.55, zorder=3)
for i, l in enumerate(labels_b):
    v = b.loc[l, "share_per_feature"]
    ax.text(v + 0.12, yy3[i], f"{v:.2f}%", va="center", fontsize=8.5, color=INK)
    ax.text(0.12, yy3[i] - 0.36,
            f"{int(b.loc[l,'n_features'])} features, {b.loc[l,'total_share']:.0f}% of total",
            va="center", fontsize=7.2, color="white" if v > 2 else MUTED)
ax.set_yticks(yy3); ax.set_yticklabels([wrap(l, 12) for l in labels_b], fontsize=8.5)
ax.set_xlim(0, max(b.share_per_feature) * 1.32)
ax.set_xlabel("mean share of |SHAP| per feature (%)")
ax.set_title("c  Attribution per feature\n(the archive is 4x denser\nin signal)", loc="left")
fig.savefig(FIG / "Figure4_shap.png")
plt.close(fig)
print("  -> Figures/Figure4_shap.png")

# ---------------------------------------------------------------- Figure 5  residuals
fig = plt.figure(figsize=(13.6, 4.9))
gs = fig.add_gridspec(1, 3, width_ratios=[1.05, 0.95, 1.0], wspace=0.58)

ax = fig.add_subplot(gs[0, 0])
lim = np.percentile(np.abs(D.residual), 97)
sc = ax.scatter(D.lon, D.lat, c=D.residual, cmap="RdBu_r", s=17, vmin=-lim, vmax=lim,
                linewidths=0.25, edgecolor="white", zorder=3)
hi_pts = D[D.resid_cluster == "high-error cluster"]
if len(hi_pts):
    ax.scatter(hi_pts.lon, hi_pts.lat, facecolor="none", edgecolor="#111", s=64,
               linewidth=1.1, zorder=4, label=f"high-error cluster (n={len(hi_pts)})")
    ax.legend(fontsize=8, loc="lower right")
else:
    ax.text(0.02, 0.02, "no high-error cluster survives\nFDR correction", transform=ax.transAxes,
            fontsize=7.6, color=MUTED, style="italic", va="bottom")
cb = fig.colorbar(sc, ax=ax, pad=0.02, fraction=0.045)
cb.set_label("residual", fontsize=8); cb.outline.set_visible(False)
cb.ax.tick_params(labelsize=7.5)
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title("a  Where the model errs\n"
             f"Moran's I: target {mi_raw.I:+.3f} → residual {mi_resid.I:+.3f}\n"
             f"({RESULTS['moran_structure_removed_pct']:.0f}% of spatial structure removed, "
             f"p = {mi_resid.p_sim:.3f})", loc="left", fontsize=9.5)
ax.set_aspect("equal", adjustable="datalim")

ax = fig.add_subplot(gs[0, 1])
leg = LEG.sort_values("r2_full")
yy = np.arange(len(leg))
ax.barh(yy + 0.21, leg.r2_full, color=ACCENT, height=0.42, label="all 45 features", zorder=3)
ax.barh(yy - 0.21, leg.r2_open_data_only, color="#6b8f71", height=0.42,
        label="open data only", zorder=3)
ax.axvline(0, color=INK, lw=1.0, zorder=2)
ax.set_yticks(yy); ax.set_yticklabels(leg.issue, fontsize=8.5)
ax.set_xlabel("block-CV $R^2$")
ax.set_title("b  Which burdens can the\ndata see at all?", loc="left")
ax.legend(fontsize=8, loc="upper left")

ax = fig.add_subplot(gs[0, 2])
rho_rl = RESULTS.get("rho::residents vs data legibility")
ax.scatter(LEG.resident_mean, LEG.r2_full, s=70, color=ACCENT, zorder=4,
           edgecolor="white", linewidth=1.2)
for _, r in LEG.iterrows():
    ax.annotate(r.issue, (r.resident_mean, r.r2_full), textcoords="offset points",
                xytext=(0, 11), ha="center", fontsize=7.6, color=INK)
zz = np.polyfit(LEG.resident_mean, LEG.r2_full, 1)
xs = np.linspace(LEG.resident_mean.min() - 0.05, LEG.resident_mean.max() + 0.05, 20)
ax.plot(xs, np.polyval(zz, xs), color="#b23a48", lw=1.6, ls="--", zorder=3)
ax.axhline(0, color=MUTED, lw=0.9, zorder=2)
ax.set_xlabel("mean burden reported by residents (1–5)")
ax.set_ylabel("block-CV $R^2$ (data legibility)")
ax.set_title(f"c  The inversion\nSpearman $\\rho$ = {rho_rl:+.3f}: the burdens\n"
             f"residents feel most are the ones\nthe data can see least", loc="left", fontsize=9.5)
ax.margins(x=0.16, y=0.30)
fig.savefig(FIG / "Figure5_spatial_accountability.png")
plt.close(fig)
print("  -> Figures/Figure5_spatial_accountability.png")


  -> Figures/Figure3_model_ladder.png
  -> Tables/T14_shap_importance.csv  (45 rows)


  -> Figures/Figure4_shap.png


  -> Figures/Figure5_spatial_accountability.png


## Step 10 — RQ4: infrastructure for validating LLM coding of Vietnamese open text

**What this step does and does not do.** It builds every reproducible component of the
LLM-validation study: the coding frame, the stratified sample, the two blind coder
workbooks, the three prompt specifications, and the two-tier Cohen's kappa computation.

It does **not** produce kappa values, because the gold standard requires two human coders
working independently. `Coding/kappa_report.py` computes the full table the moment the two
returned workbooks are placed in `Coding/returned/`. Until then the manuscript's Table 20
carries the protocol and empty result cells, and RQ4 is declared as protocol-registered
rather than completed. Reporting invented kappa values would be worse than reporting none.


In [20]:
OPEN_TEXT_FIELDS = [
    ("Citizens", "Q2.8_LyDo_KSD_PTCC",   "Reason for not using public transport"),
    ("Citizens", "Q3.1_LyDo_ChonNoiSong", "Reason for choosing this place to live"),
    ("Citizens", "Q4.4_MongMuon_Tgia_PTDT", "Desired form of participation in urban development"),
    ("Citizens", "Q4.5_MongMuon_Tgia_CS", "Desired form of participation in policy"),
    ("Citizens", "Q4.10_MongMuon_ThayDoi", "Desired change in the area"),
    ("SMEs",     "Q3.1_LyDo_ChonNoiKD",   "Reason for choosing this business location"),
    ("SMEs",     "Q4.6_ThachThuc",        "Main business challenge"),
    ("SMEs",     "Q4.12_MM_ThayDoi",      "Desired change in the area"),
]

frames = {"Citizens": CIT, "SMEs": SME}
corpus = []
for inst, col, desc in OPEN_TEXT_FIELDS:
    df = frames[inst]
    if col not in df.columns:
        print(f"  ! missing column {inst}.{col}")
        continue
    s = df[col].dropna().astype(str).str.strip()
    s = s[(s.str.len() >= 8) & (~s.str.lower().isin(["không", "khong", "không có", "n/a", "na", "-"]))]
    for i, v in s.items():
        corpus.append({"instrument": inst, "field": col, "field_desc": desc,
                       "row_id": int(i), "text": v, "n_chars": len(v)})
CORPUS = pd.DataFrame(corpus)
print(f"open-text corpus: {len(CORPUS)} segments across {CORPUS.field.nunique()} fields")
print(CORPUS.groupby(["instrument", "field_desc"]).agg(
    n=("text", "size"), mean_chars=("n_chars", "mean")).round(1).to_string())
rec("corpus_n_segments", int(len(CORPUS)))
rec("corpus_n_fields", int(CORPUS.field.nunique()))
rec("corpus_mean_chars", round(float(CORPUS.n_chars.mean()), 1))
CORPUS.to_csv(COD / "corpus_full.csv", index=False, encoding="utf-8-sig")

# ---------------------------------------------------------------- coding frame
CODING_FRAME = [
    ("INFRA",  "Infrastructure and physical environment",
     "roads, drainage, flooding, buildings, utilities, public space"),
    ("MOBIL",  "Mobility and transport",
     "traffic, congestion, distance, travel time, vehicles, public transport"),
    ("ENVIR",  "Environmental quality",
     "air, noise, water, waste, greenery, heat"),
    ("ECON",   "Economic and livelihood",
     "cost, income, business, employment, rent, customers"),
    ("SOCIAL", "Social and community",
     "family, neighbours, community, safety, security, schooling, health services"),
    ("GOVERN", "Governance and administration",
     "policy, officials, procedures, participation, information, planning"),
    ("HABIT",  "Habit, preference and convenience",
     "used to it, convenient, prefer, no particular reason"),
    ("OTHER",  "Other or uncodable",
     "anything not covered above, or too vague to code"),
]
FRAME = pd.DataFrame(CODING_FRAME, columns=["code", "label", "indicators"])
print("\ncoding frame:")
print(FRAME.to_string(index=False))
FRAME.to_csv(COD / "coding_frame.csv", index=False, encoding="utf-8-sig")
rec("coding_frame_n_codes", len(FRAME))

# ---------------------------------------------------------------- sample
N_GOLD = 200
rg = np.random.default_rng(RNG)
# stratify proportionally by field, minimum 12 per field
alloc = (CORPUS.field.value_counts(normalize=True) * N_GOLD).round().astype(int).clip(lower=12)
alloc = (alloc * N_GOLD / alloc.sum()).round().astype(int)
parts = []
for f, k in alloc.items():
    sub = CORPUS[CORPUS.field == f]
    parts.append(sub.sample(n=min(k, len(sub)), random_state=RNG))
SAMPLE = pd.concat(parts).sample(frac=1, random_state=RNG).reset_index(drop=True)
SAMPLE.insert(0, "seg_id", [f"S{i:03d}" for i in range(1, len(SAMPLE) + 1)])
print(f"\ngold-standard sample: {len(SAMPLE)} segments")
print(SAMPLE.field_desc.value_counts().to_string())
rec("gold_sample_n", int(len(SAMPLE)))
SAMPLE.to_csv(COD / "gold_sample.csv", index=False, encoding="utf-8-sig")

# ---------------------------------------------------------------- coder workbooks
try:
    from openpyxl import Workbook
    from openpyxl.styles import Font, Alignment, PatternFill
    from openpyxl.worksheet.datavalidation import DataValidation

    for coder in ["A", "B"]:
        wb = Workbook()
        ws = wb.active; ws.title = "coding"
        ws.append(["seg_id", "instrument", "question", "text (Vietnamese)",
                   "primary_code", "secondary_code (optional)", "confidence 1-3", "note"])
        for c in ws[1]:
            c.font = Font(bold=True, color="FFFFFF")
            c.fill = PatternFill("solid", fgColor="1B4965")
            c.alignment = Alignment(vertical="center", wrap_text=True)
        for _, r in SAMPLE.iterrows():
            ws.append([r.seg_id, r.instrument, r.field_desc, r.text, "", "", "", ""])
        widths = {"A": 9, "B": 11, "C": 34, "D": 62, "E": 15, "F": 22, "G": 15, "H": 30}
        for col, w in widths.items():
            ws.column_dimensions[col].width = w
        for row in ws.iter_rows(min_row=2):
            row[3].alignment = Alignment(wrap_text=True, vertical="top")
        dv = DataValidation(type="list", formula1='"' + ",".join(FRAME.code) + '"', allow_blank=True)
        ws.add_data_validation(dv); dv.add(f"E2:F{len(SAMPLE)+1}")
        dv2 = DataValidation(type="list", formula1='"1,2,3"', allow_blank=True)
        ws.add_data_validation(dv2); dv2.add(f"G2:G{len(SAMPLE)+1}")
        ws.freeze_panes = "A2"

        ws2 = wb.create_sheet("coding frame")
        ws2.append(["code", "label", "indicators"])
        for c in ws2[1]:
            c.font = Font(bold=True, color="FFFFFF")
            c.fill = PatternFill("solid", fgColor="1B4965")
        for _, r in FRAME.iterrows():
            ws2.append([r.code, r.label, r.indicators])
        for col, w in {"A": 10, "B": 40, "C": 66}.items():
            ws2.column_dimensions[col].width = w

        ws3 = wb.create_sheet("instructions")
        for line in [
            f"CODER {coder} — TTLL Paper 4, open-response coding",
            "",
            "1. Assign exactly one primary_code to every segment. Use the drop-down in column E.",
            "2. Add a secondary_code only when a segment genuinely carries a second theme.",
            "3. Record your confidence 1 (low) to 3 (high) in column G.",
            "4. Code independently. Do NOT discuss any segment with the other coder,",
            "   and do not look at the other coder's workbook, until both are returned.",
            "5. Use OTHER sparingly, and write why in the note column when you do.",
            "6. Return the file unchanged in structure to Coding/returned/coder_" + coder + ".xlsx",
            "",
            "Only after both workbooks are returned is inter-coder agreement computed",
            "(kappa_report.py), disagreements adjudicated, and the gold standard fixed.",
            "The LLM comparison runs only against the fixed gold standard.",
        ]:
            ws3.append([line])
        ws3.column_dimensions["A"].width = 100
        wb.save(COD / f"coder_{coder}_blank.xlsx")
        print(f"  -> Coding/coder_{coder}_blank.xlsx")
except ImportError:
    print("openpyxl unavailable - coder workbooks not written")

(COD / "returned").mkdir(exist_ok=True)


open-text corpus: 2533 segments across 8 fields
                                                                 n  mean_chars
instrument field_desc                                                         
Citizens   Desired change in the area                          457        45.6
           Desired form of participation in policy             460        26.5
           Desired form of participation in urban development  460        27.4
           Reason for choosing this place to live              457        34.1
           Reason for not using public transport               460        41.6
SMEs       Desired change in the area                           80        40.0
           Main business challenge                              79        49.3
           Reason for choosing this business location           80        61.0

coding frame:
  code                                   label                                                                  indicators
 INFRA Infrastructure an

### 10.2 — Prompt specifications and the kappa report script

Three prompting strategies, written to disk verbatim so that they are citable in the
Supplementary Material. The model identifier, access date, temperature and the number of
repeated runs are recorded in the run manifest, not hard-coded here, because the point of
the exercise is reproducibility under model drift.


In [21]:
CODES_BLOCK = "\n".join(f"- {r.code}: {r.label} ({r.indicators})" for _, r in FRAME.iterrows())

PROMPTS = {
"zero_shot": f"""You are coding open-ended survey responses written in Vietnamese by residents
and small business owners in Truong Tho ward, Thu Duc City, Ho Chi Minh City.

Assign exactly ONE primary code from this frame:
{CODES_BLOCK}

Rules:
- Answer with the code only, nothing else.
- Use OTHER only when no other code applies.
- Do not translate, explain, or paraphrase.

QUESTION: {{question}}
RESPONSE: {{text}}
CODE:""",

"few_shot": f"""You are coding open-ended survey responses written in Vietnamese by residents
and small business owners in Truong Tho ward, Thu Duc City, Ho Chi Minh City.

Assign exactly ONE primary code from this frame:
{CODES_BLOCK}

Examples:
RESPONSE: "Đường ngập nước mỗi khi mưa lớn" -> INFRA
RESPONSE: "Gần chỗ làm, đi lại thuận tiện" -> MOBIL
RESPONSE: "Không khí ô nhiễm, nhiều bụi" -> ENVIR
RESPONSE: "Giá thuê mặt bằng cao quá" -> ECON
RESPONSE: "Ở đây quen rồi, không muốn chuyển" -> HABIT
RESPONSE: "Thủ tục hành chính rườm rà" -> GOVERN

Answer with the code only, nothing else.

QUESTION: {{question}}
RESPONSE: {{text}}
CODE:""",

"chain_of_thought": f"""You are coding open-ended survey responses written in Vietnamese by
residents and small business owners in Truong Tho ward, Thu Duc City, Ho Chi Minh City.

Coding frame:
{CODES_BLOCK}

Work through these steps, then give the code.
STEP 1 - What is the response literally about? (one short clause)
STEP 2 - Which two codes are the closest candidates?
STEP 3 - Which of the two fits better, and why? (one short clause)
FINAL - output exactly: CODE: <code>

QUESTION: {{question}}
RESPONSE: {{text}}
""",
}
for name, text in PROMPTS.items():
    (COD / f"prompt_{name}.txt").write_text(text, encoding="utf-8")
    print(f"  -> Coding/prompt_{name}.txt ({len(text)} chars)")
rec("llm_prompt_strategies", list(PROMPTS))

RUN_MANIFEST = {
    "status": "PENDING - awaiting human gold standard",
    "sample_size": int(len(SAMPLE)),
    "coding_frame_codes": list(FRAME.code),
    "prompt_strategies": list(PROMPTS),
    "planned_repeats_per_strategy": 3,
    "temperature": 0.0,
    "models_to_run": [
        {"role": "frontier API model", "identifier": "TO BE RECORDED AT RUN TIME",
         "access_date": None},
        {"role": "locally hosted open-weights anchor",
         "identifier": "TO BE RECORDED AT RUN TIME (e.g. a Qwen or Llama instruct model "
                       "served locally, so the comparison remains reproducible after API drift)",
         "access_date": None},
    ],
    "note": "Every field marked TO BE RECORDED must be filled before the manuscript is "
            "submitted; the Supplementary Material reproduces this manifest verbatim.",
}
with open(COD / "run_manifest.json", "w", encoding="utf-8") as fh:
    json.dump(RUN_MANIFEST, fh, ensure_ascii=False, indent=2)
print("  -> Coding/run_manifest.json")

# ---------------------------------------------------------------- kappa report script
KAPPA_SCRIPT = r'''#!/usr/bin/env python3
"""Two-tier Cohen's kappa for TTLL Paper 4, RQ4.

Tier 1  human A vs human B          -> establishes that the frame is codeable at all
Tier 2  LLM vs the adjudicated gold -> the claim the paper actually makes

Run:  python kappa_report.py
Reads:  returned/coder_A.xlsx, returned/coder_B.xlsx, and any llm_<strategy>_run<k>.csv
Writes: kappa_results.csv, gold_standard.csv, disagreements.csv
"""
import sys, glob, json
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix

HERE = Path(__file__).resolve().parents[2] / "Coding"
FRAME = pd.read_csv(HERE / "coding_frame.csv")
CODES = list(FRAME.code)
SAMPLE = pd.read_csv(HERE / "gold_sample.csv")


def boot_kappa(a, b, n=2000, seed=7):
    rg = np.random.default_rng(seed); idx = np.arange(len(a)); out = []
    for _ in range(n):
        s = rg.choice(idx, len(idx), replace=True)
        if len(set(a[s])) < 2 or len(set(b[s])) < 2:
            continue
        out.append(cohen_kappa_score(a[s], b[s], labels=CODES))
    return float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))


def load_coder(p):
    d = pd.read_excel(p, sheet_name="coding")
    d = d[["seg_id", "primary_code"]].dropna()
    d["primary_code"] = d.primary_code.astype(str).str.strip().str.upper()
    return d


ret = HERE / "returned"
pa, pb = ret / "coder_A.xlsx", ret / "coder_B.xlsx"
if not (pa.exists() and pb.exists()):
    sys.exit("Both returned/coder_A.xlsx and returned/coder_B.xlsx are required. "
             "RQ4 cannot be reported until human coding is complete.")

A, B = load_coder(pa), load_coder(pb)
M = A.merge(B, on="seg_id", suffixes=("_A", "_B"))
print(f"segments coded by both: {len(M)}")

rows = []
k_hh = cohen_kappa_score(M.primary_code_A, M.primary_code_B, labels=CODES)
lo, hi = boot_kappa(M.primary_code_A.values, M.primary_code_B.values)
agree = (M.primary_code_A == M.primary_code_B).mean()
rows.append({"tier": "human-human", "comparison": "coder A vs coder B", "n": len(M),
             "raw_agreement": round(agree, 4), "kappa": round(k_hh, 4),
             "ci_low": round(lo, 4), "ci_high": round(hi, 4)})
print(f"TIER 1  human-human kappa = {k_hh:.3f} [{lo:.3f}, {hi:.3f}], "
      f"raw agreement {agree:.1%}")
if k_hh < 0.60:
    print("  WARNING: kappa below 0.60. Revise the coding frame and recode before "
          "any LLM comparison. Do not proceed.")

dis = M[M.primary_code_A != M.primary_code_B].merge(SAMPLE[["seg_id", "text", "field_desc"]],
                                                    on="seg_id", how="left")
dis.to_csv(HERE / "disagreements.csv", index=False, encoding="utf-8-sig")
print(f"  -> disagreements.csv ({len(dis)} segments for adjudication)")

gold_path = HERE / "gold_standard.csv"
if gold_path.exists():
    GOLD = pd.read_csv(gold_path)[["seg_id", "gold_code"]]
    print(f"adjudicated gold standard loaded: {len(GOLD)} segments")
else:
    GOLD = M[M.primary_code_A == M.primary_code_B][["seg_id", "primary_code_A"]]
    GOLD.columns = ["seg_id", "gold_code"]
    GOLD.to_csv(gold_path, index=False, encoding="utf-8-sig")
    print(f"  -> gold_standard.csv seeded with the {len(GOLD)} agreed segments. "
          f"Adjudicate disagreements.csv, append them, and re-run.")

for f in sorted(glob.glob(str(HERE / "llm_*.csv"))):
    name = Path(f).stem
    L = pd.read_csv(f)[["seg_id", "code"]]
    L["code"] = L.code.astype(str).str.strip().str.upper()
    J = GOLD.merge(L, on="seg_id")
    if len(J) < 20:
        continue
    k = cohen_kappa_score(J.gold_code, J.code, labels=CODES)
    lo, hi = boot_kappa(J.gold_code.values, J.code.values)
    rows.append({"tier": "llm-gold", "comparison": name, "n": len(J),
                 "raw_agreement": round((J.gold_code == J.code).mean(), 4),
                 "kappa": round(k, 4), "ci_low": round(lo, 4), "ci_high": round(hi, 4)})
    print(f"TIER 2  {name:32s} kappa = {k:.3f} [{lo:.3f}, {hi:.3f}]")
    cm = pd.DataFrame(confusion_matrix(J.gold_code, J.code, labels=CODES),
                      index=CODES, columns=CODES)
    cm.to_csv(HERE / f"confusion_{name}.csv", encoding="utf-8-sig")

OUT = pd.DataFrame(rows)
OUT.to_csv(HERE / "kappa_results.csv", index=False, encoding="utf-8-sig")
print("\n" + OUT.to_string(index=False))
print("\n  -> kappa_results.csv   (paste into manuscript Table 20)")
'''
(P4 / "code-p4" / "Coding" / "kappa_report.py").write_text(KAPPA_SCRIPT, encoding="utf-8")
print("  -> Coding/kappa_report.py")

RUNNER = r'''#!/usr/bin/env python3
"""Run the three prompt strategies over gold_sample.csv and write llm_<strategy>_run<k>.csv.

Deliberately model-agnostic: implement call_model() for whichever endpoint you use, record
the identifier and access date in run_manifest.json, and run each strategy 3 times so the
manuscript can report between-run variance rather than a single lucky pass.
"""
import json, time, sys
from pathlib import Path
import pandas as pd

HERE = Path(__file__).resolve().parents[2] / "Coding"
SAMPLE = pd.read_csv(HERE / "gold_sample.csv")
STRATEGIES = ["zero_shot", "few_shot", "chain_of_thought"]
N_REPEATS = 3
VALID = set(pd.read_csv(HERE / "coding_frame.csv").code)


def call_model(prompt: str) -> str:
    """IMPLEMENT ME. Return the raw model output for one prompt.

    Keep temperature at 0.0 and record the exact model identifier in run_manifest.json.
    Example shapes:
      - Anthropic:  client.messages.create(model=..., max_tokens=64, temperature=0,
                                           messages=[{"role":"user","content":prompt}])
      - local HF:   pipe(prompt, max_new_tokens=64, do_sample=False)[0]["generated_text"]
    """
    raise NotImplementedError("Implement call_model() for your endpoint before running.")


def parse(out: str) -> str:
    up = out.strip().upper()
    if "CODE:" in up:
        up = up.split("CODE:")[-1]
    for tok in up.replace("\n", " ").split():
        tok = tok.strip(".,:;*`\"'")
        if tok in VALID:
            return tok
    return "UNPARSED"


for strat in STRATEGIES:
    tpl = (HERE / f"prompt_{strat}.txt").read_text(encoding="utf-8")
    for run in range(1, N_REPEATS + 1):
        out_path = HERE / f"llm_{strat}_run{run}.csv"
        if out_path.exists():
            print(f"skip {out_path.name} (already present)"); continue
        rows = []
        for _, r in SAMPLE.iterrows():
            prompt = tpl.replace("{question}", str(r.field_desc)).replace("{text}", str(r.text))
            raw = call_model(prompt)
            rows.append({"seg_id": r.seg_id, "code": parse(raw), "raw": raw})
            time.sleep(0.1)
        pd.DataFrame(rows).to_csv(out_path, index=False, encoding="utf-8-sig")
        print(f"  -> {out_path.name}")
print("\nNow run kappa_report.py.")
'''
(P4 / "code-p4" / "Coding" / "run_llm_coding.py").write_text(RUNNER, encoding="utf-8")
print("  -> Coding/run_llm_coding.py")

T20 = pd.DataFrame([
    {"tier": "Tier 1", "comparison": "Coder A vs Coder B (human-human)",
     "n": len(SAMPLE), "kappa": "pending", "ci": "pending",
     "purpose": "establishes that the coding frame is applicable at all"},
    {"tier": "Tier 2", "comparison": "LLM zero-shot vs adjudicated gold",
     "n": len(SAMPLE), "kappa": "pending", "ci": "pending",
     "purpose": "baseline LLM performance"},
    {"tier": "Tier 2", "comparison": "LLM few-shot vs adjudicated gold",
     "n": len(SAMPLE), "kappa": "pending", "ci": "pending",
     "purpose": "effect of in-context examples"},
    {"tier": "Tier 2", "comparison": "LLM chain-of-thought vs adjudicated gold",
     "n": len(SAMPLE), "kappa": "pending", "ci": "pending",
     "purpose": "effect of explicit reasoning"},
    {"tier": "Tier 2", "comparison": "Local open-weights model vs adjudicated gold",
     "n": len(SAMPLE), "kappa": "pending", "ci": "pending",
     "purpose": "reproducible anchor that survives API model drift"},
])
save_table(T20, "T20_llm_validation_protocol",
           "Two-tier validation protocol for LLM coding of Vietnamese open responses. "
           "Values are pending completion of independent human coding.")
rec("rq4_status", "protocol registered; awaiting two-coder gold standard")
print("\nRQ4 status: infrastructure complete, results pending human coding.")


  -> Coding/prompt_zero_shot.txt (1164 chars)
  -> Coding/prompt_few_shot.txt (1382 chars)
  -> Coding/prompt_chain_of_thought.txt (1254 chars)
  -> Coding/run_manifest.json
  -> Coding/kappa_report.py
  -> Coding/run_llm_coding.py
  -> Tables/T20_llm_validation_protocol.csv  (5 rows)

RQ4 status: infrastructure complete, results pending human coding.


## Step 11 — Figure 1: the pipeline architecture

The most important figure in the paper. Each layer is annotated with the specific data
friction it removes and with the concrete quantity this study produced at that layer, so
the diagram is a record of what ran rather than a proposal for what might.


In [22]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

LAYERS = [
    ("L1", "Harmonisation",
     "four instruments → one schema",
     f"{RESULTS['n_citizens']} + {RESULTS['n_smes']} + {RESULTS['n_experts']} + "
     f"{RESULTS['n_authorities']} records;\n3 Likert families reconciled;\n"
     "machine-readable codebook emitted",
     "friction removed: incompatible instruments"),
    ("L2", "Quality assurance",
     "automated gates, run before and after enrichment",
     f"{RESULTS['qa_flags_total']} flags raised "
     f"({RESULTS['qa_out_of_scale']} out-of-scale, {RESULTS['qa_bad_coords']} bad coordinate);\n"
     f"α = {RESULTS['alpha_burden']}, KMO = {RESULTS['kmo']},\n"
     f"Moran's I = {RESULTS['moran_flood']:+.3f} re-verified",
     "friction removed: silent construct drift"),
    ("L3", "Enrichment",
     "spatial join to free open data",
     f"{RESULTS['n_open_features']} features from Copernicus DEM, Sentinel-1/2,\n"
     f"Landsat LST, GHSL, JRC GSW, OpenStreetMap;\n"
     f"{RESULTS['p1_features_full_coverage']}/{RESULTS['n_open_features']} at 100% point coverage",
     "friction removed: manual GIS handoffs"),
    ("L4", "Inference",
     "interpretable models + explanation + text coding",
     f"8-rung ladder under random AND spatial block CV;\n"
     f"SHAP over 45 features; residual diagnostics;\n"
     f"LLM coding protocol on {RESULTS['corpus_n_segments']} open-text segments",
     "friction removed: unauditable black boxes"),
    ("L5", "Decision interface",
     "spatial dashboard with an uncertainty layer",
     "burden surface, model residual, stakeholder\n"
     "divergence, and per-burden data legibility;\n"
     "reports where it should not be trusted",
     "friction removed: results that never reach a decision"),
]
LCOL = ["#1b4965", "#2a6f97", "#468faf", "#61a5c2", "#89c2d9"]

fig, ax = plt.subplots(figsize=(13.6, 8.4))
ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis("off")

# input band
ax.add_patch(FancyBboxPatch((3, 88), 94, 9, boxstyle="round,pad=0.6,rounding_size=1.2",
                            facecolor="#eef2f5", edgecolor="#c9d3da", linewidth=1.0))
ax.text(50, 94.6, "LEGACY STAKEHOLDER ARCHIVE  +  OPEN DATA", ha="center", va="center",
        fontsize=11, fontweight="bold", color="#22252a")
ax.text(50, 90.6,
        f"4 questionnaires collected 2023 and shelved  ·  {RESULTS['n_citizens']} residents, "
        f"{RESULTS['n_smes']} SMEs, {RESULTS['n_experts']} experts, {RESULTS['n_authorities']} officials"
        f"   |   {RESULTS['n_open_features']} freely available geospatial features",
        ha="center", va="center", fontsize=8.6, color="#4a545c")

top, h, gap = 84.0, 13.4, 2.4
for i, (code, name, sub, detail, friction) in enumerate(LAYERS):
    y0 = top - (i + 1) * (h + gap) + gap
    ax.add_patch(FancyBboxPatch((3, y0), 94, h, boxstyle="round,pad=0.5,rounding_size=1.2",
                                facecolor="white", edgecolor=LCOL[i], linewidth=1.8))
    ax.add_patch(FancyBboxPatch((3, y0), 13.5, h, boxstyle="round,pad=0.5,rounding_size=1.2",
                                facecolor=LCOL[i], edgecolor=LCOL[i], linewidth=1.8))
    ax.text(9.7, y0 + h * 0.62, code, ha="center", va="center", fontsize=17,
            fontweight="bold", color="white")
    ax.text(9.7, y0 + h * 0.24, ["harmonise", "verify", "enrich", "infer", "decide"][i],
            ha="center", va="center", fontsize=7.4, color="white", alpha=0.88, style="italic")
    ax.text(18.5, y0 + h * 0.76, name, ha="left", va="center", fontsize=12.5,
            fontweight="bold", color=LCOL[i])
    ax.text(18.5, y0 + h * 0.50, sub, ha="left", va="center", fontsize=8.6,
            color="#4a545c", style="italic")
    ax.text(18.5, y0 + h * 0.20, friction, ha="left", va="center", fontsize=7.8, color="#8a949c")
    ax.text(96, y0 + h * 0.50, detail, ha="right", va="center", fontsize=8.0, color="#22252a",
            linespacing=1.45)
    if i < len(LAYERS) - 1:
        ax.add_patch(FancyArrowPatch((50, y0 - 0.4), (50, y0 - gap + 0.5),
                                     arrowstyle="-|>", mutation_scale=14,
                                     color="#9aa0a6", linewidth=1.3))

y0 = top - 5 * (h + gap) + gap
ax.add_patch(FancyArrowPatch((50, y0 - 0.7), (50, y0 - 2.4), arrowstyle="-|>",
                             mutation_scale=14, color="#9aa0a6", linewidth=1.3))
ax.add_patch(FancyBboxPatch((4, 0.2), 92, 4.6, boxstyle="round,pad=0.5,rounding_size=1.2",
                            facecolor="#fdf3ec", edgecolor="#e0b494", linewidth=1.2))
ax.text(50, 2.5,
        "WARD-SCALE DECISION SUPPORT   ·   every component free and open   ·   "
        "reproducible from four spreadsheets and a public feature stack",
        ha="center", va="center", fontsize=9.0, fontweight="bold", color="#8a4b1e")

# the feedback loop from L2
ax.annotate("", xy=(1.6, top - (h + gap) + gap + h * 0.5), xytext=(1.6, top - 3 * (h + gap) + gap + h * 0.5),
            arrowprops=dict(arrowstyle="-|>", color="#c26a3d", lw=1.3,
                            connectionstyle="arc3,rad=-0.55"))
ax.text(0.3, top - 2 * (h + gap) + gap + h * 0.5, "L2 re-runs\nafter L3", rotation=90,
        ha="center", va="center", fontsize=7.2, color="#c26a3d", fontweight="bold")

fig.savefig(FIG / "Figure1_pipeline.png")
plt.close(fig)
print("  -> Figures/Figure1_pipeline.png")


  -> Figures/Figure1_pipeline.png


## Step 12 — Layer 5: the decision interface

A self-contained HTML dashboard is written to `Dashboard/ttll_decision_dashboard.html`.
It embeds its own data, needs no server and no network, and opens in any browser — which
matters, because a ward office that cannot install software is exactly the deployment
context the paper is arguing about.

The design commitment is that the interface reports its own unreliability: the residual
layer and the per-burden legibility panel are not diagnostics hidden in an appendix, they
are first-class views next to the prediction.


In [23]:
dash_pts = D[["lat", "lon", "burden_mean", "pred", "residual", "ward", "block"]].copy()
dash_pts = dash_pts.round({"lat": 6, "lon": 6, "burden_mean": 3, "pred": 3, "residual": 3})
PAYLOAD = {
    "generated": "TTLL Paper 4 pipeline, Layer 5",
    "points": dash_pts.to_dict("records"),
    "legibility": LEG[["issue", "resident_mean", "expert_severity",
                       "authority_agenda_pct", "r2_full", "r2_open_data_only",
                       "morans_I"]].round(4).to_dict("records"),
    "ladder": T11[["model", "scheme", "r2", "r2_lo", "r2_hi", "mae"]].to_dict("records"),
    "shap": SHAPIMP.head(12)[["label", "share_pct", "block"]].to_dict("records"),
    "barriers": T1[["barrier_en", "class", "pct", "ci_low", "ci_high"]].to_dict("records"),
    "stats": {
        "n_points": int(len(dash_pts)),
        "best_model": BEST,
        "best_r2": float(RESULTS["best_block_r2"]),
        "best_r2_lo": float(RESULTS["best_block_r2_ci"][0]),
        "best_r2_hi": float(RESULTS["best_block_r2_ci"][1]),
        "best_mae": float(RESULTS["best_block_mae"]),
        "moran_target": float(RESULTS["moran_burden_mean"]),
        "moran_resid": float(RESULTS["moran_residual"]),
        "alpha": float(RESULTS["alpha_burden"]),
        "rho_invert": float(RESULTS["rho::residents vs data legibility"]),
        "eai_paper2": 0.843,
        "dt_expert": float(RESULTS["dt_progress_expert_mean"]),
        "dt_auth": float(RESULTS["dt_progress_auth_mean"]),
        "tools_expert": float(RESULTS["eval_tools_expert_agree_pct"]),
        "tools_auth": float(RESULTS["eval_tools_auth_agree_pct"]),
    },
}

HTML = """<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>TTLL Decision Dashboard — Truong Tho Living Lab</title>
<style>
:root{--ink:#1a1d21;--mut:#6b7480;--line:#dde3e8;--bg:#f6f8fa;--card:#fff;
      --acc:#1b4965;--acc2:#468faf;--warn:#c26a3d;--bad:#b23a48;--ok:#4a7c59}
*{box-sizing:border-box}
body{margin:0;font:14px/1.5 -apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif;
     background:var(--bg);color:var(--ink)}
header{background:var(--acc);color:#fff;padding:18px 26px}
header h1{margin:0;font-size:19px;letter-spacing:-.2px}
header p{margin:5px 0 0;opacity:.82;font-size:12.5px}
.wrap{max-width:1320px;margin:0 auto;padding:20px 26px 50px}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:12px;margin-bottom:18px}
.kpi{background:var(--card);border:1px solid var(--line);border-radius:9px;padding:13px 15px}
.kpi .lab{font-size:10.5px;text-transform:uppercase;letter-spacing:.7px;color:var(--mut)}
.kpi .val{font-size:24px;font-weight:650;margin-top:3px;letter-spacing:-.5px}
.kpi .sub{font-size:11px;color:var(--mut);margin-top:2px}
.grid{display:grid;grid-template-columns:1.25fr 1fr;gap:16px}
@media(max-width:960px){.grid{grid-template-columns:1fr}}
.card{background:var(--card);border:1px solid var(--line);border-radius:9px;padding:16px 18px;margin-bottom:16px}
.card h2{margin:0 0 3px;font-size:14.5px}
.card .cap{margin:0 0 12px;font-size:11.5px;color:var(--mut)}
.tabs{display:flex;gap:6px;margin-bottom:11px;flex-wrap:wrap}
.tab{padding:5px 11px;border:1px solid var(--line);border-radius:20px;background:#fff;
     cursor:pointer;font-size:12px;color:var(--mut)}
.tab[aria-selected=true]{background:var(--acc);border-color:var(--acc);color:#fff}
svg{display:block;width:100%;height:auto}
table{width:100%;border-collapse:collapse;font-size:12.2px}
th,td{padding:6px 7px;border-bottom:1px solid var(--line);text-align:right}
th:first-child,td:first-child{text-align:left}
th{font-weight:600;color:var(--mut);font-size:10.5px;text-transform:uppercase;letter-spacing:.5px}
.bar{height:9px;border-radius:5px;background:var(--acc2);display:inline-block;vertical-align:middle}
.note{background:#fdf3ec;border:1px solid #e8c6a8;border-left:4px solid var(--warn);
      border-radius:7px;padding:12px 15px;font-size:12.5px;margin-bottom:16px}
.note strong{color:#8a4b1e}
.legend{display:flex;gap:14px;flex-wrap:wrap;font-size:11.5px;color:var(--mut);margin-top:9px}
.sw{width:11px;height:11px;border-radius:3px;display:inline-block;margin-right:5px;vertical-align:-1px}
.tip{position:fixed;pointer-events:none;background:#1a1d21;color:#fff;padding:6px 9px;
     border-radius:6px;font-size:11.5px;opacity:0;transition:opacity .1s;z-index:9}
footer{color:var(--mut);font-size:11.5px;margin-top:8px;text-align:center}
</style></head><body>
<header>
  <h1>Truong Tho Living Lab — Decision Dashboard</h1>
  <p>Layer 5 of the reactivation pipeline · built entirely from a shelved 2023 survey archive and free open data</p>
</header>
<div class="wrap">
  <div class="note">
    <strong>This interface reports where it should not be trusted.</strong>
    The prediction layer is only one of four views. The residual layer shows where the model
    errs; the legibility panel shows which burdens the available data cannot see at all.
    Predictions for a burden with a negative legibility score should not be used to allocate anything.
  </div>
  <div class="kpis" id="kpis"></div>
  <div class="grid">
    <div>
      <div class="card">
        <h2>Ward surface</h2>
        <p class="cap" id="mapcap"></p>
        <div class="tabs" id="maptabs"></div>
        <svg id="map" viewBox="0 0 620 470"></svg>
        <div class="legend" id="maplegend"></div>
      </div>
    </div>
    <div>
      <div class="card">
        <h2>Can the data see this burden?</h2>
        <p class="cap">Block-CV R² per burden. Negative means the full 45-feature model
           predicts it worse than the sample mean.</p>
        <svg id="leg" viewBox="0 0 470 250"></svg>
      </div>
      <div class="card">
        <h2>Where the signal comes from</h2>
        <p class="cap">Share of total |SHAP| across the 12 most important features.</p>
        <svg id="shap" viewBox="0 0 470 300"></svg>
        <div class="legend">
          <span><i class="sw" style="background:#1b4965"></i>Survey archive</span>
          <span><i class="sw" style="background:#c26a3d"></i>Demographics</span>
          <span><i class="sw" style="background:#6b8f71"></i>Open data</span>
        </div>
      </div>
    </div>
  </div>
  <div class="card">
    <h2>Stakeholder divergence</h2>
    <p class="cap">Four sources of knowledge over the same six burdens. Ranks that disagree
       are the places a decision needs a human, not a model.</p>
    <table id="divtable"></table>
  </div>
  <footer>Generated by the TTLL Paper 4 analysis notebook · every number recomputed from source at build time</footer>
</div>
<div class="tip" id="tip"></div>
<script>
const DATA = __PAYLOAD__;
const $ = s => document.querySelector(s);
const tip = $("#tip");
function showTip(e, html){tip.innerHTML=html;tip.style.opacity=1;
  tip.style.left=(e.clientX+13)+"px";tip.style.top=(e.clientY-10)+"px";}
function hideTip(){tip.style.opacity=0;}
const S = DATA.stats;

// ---------------- KPIs
const kpis=[
 ["Respondents reactivated", S.n_points, "geolocated households, 2023 archive"],
 ["Best out-of-fold R²", S.best_r2.toFixed(3),
  "spatial block CV · 95% CI ["+S.best_r2_lo.toFixed(2)+", "+S.best_r2_hi.toFixed(2)+"]"],
 ["Mean absolute error", S.best_mae.toFixed(2), "on a 1–5 burden scale"],
 ["Residual clustering", (S.moran_resid>=0?"+":"")+S.moran_resid.toFixed(3),
  "Moran's I, down from "+(S.moran_target>=0?"+":"")+S.moran_target.toFixed(3)],
 ["Priority ↔ legibility", S.rho_invert.toFixed(2),
  "Spearman ρ — an inversion, not an alignment"],
 ["Officials vs experts on<br>whether tools exist",
  S.tools_auth.toFixed(0)+"% / "+S.tools_expert.toFixed(0)+"%", "agree effective instruments exist"],
];
$("#kpis").innerHTML = kpis.map(k=>
 `<div class="kpi"><div class="lab">${k[0]}</div><div class="val">${k[1]}</div><div class="sub">${k[2]}</div></div>`).join("");

// ---------------- map
const P = DATA.points;
const lons=P.map(p=>p.lon), lats=P.map(p=>p.lat);
const x0=Math.min(...lons), x1=Math.max(...lons), y0=Math.min(...lats), y1=Math.max(...lats);
const PAD=34, W=620, H=470;
const sx = v => PAD + (v-x0)/(x1-x0)*(W-2*PAD);
const sy = v => H-PAD - (v-y0)/(y1-y0)*(H-2*PAD);
const MODES = [
  {key:"burden_mean", label:"Reported burden", cap:"What residents reported in 2023 (1–5).",
   dom:[1,4.2], type:"seq"},
  {key:"pred", label:"Model prediction", cap:"Out-of-fold prediction under spatial block CV.",
   dom:[1,4.2], type:"seq"},
  {key:"residual", label:"Residual", cap:"Observed minus predicted. Read this before the prediction.",
   dom:null, type:"div"},
];
let mode = 0;
function seqColor(t){const c=[[237,242,246],[110,163,190],[27,73,101]];
  const i=t<.5?0:1, f=t<.5?t*2:(t-.5)*2;
  const a=c[i], b=c[i+1];
  return `rgb(${a.map((v,k)=>Math.round(v+(b[k]-v)*f)).join(",")})`;}
function divColor(t){ // t in [-1,1]
  const neg=[27,73,101], mid=[245,246,247], pos=[178,58,72];
  const a = t<0?neg:pos, f=Math.min(1,Math.abs(t));
  return `rgb(${mid.map((v,k)=>Math.round(v+(a[k]-v)*f)).join(",")})`;}
function drawMap(){
  const m = MODES[mode];
  const vals = P.map(p=>p[m.key]);
  let lo,hi;
  if(m.type==="div"){hi=Math.max(...vals.map(Math.abs)); lo=-hi;}
  else {lo=m.dom[0]; hi=m.dom[1];}
  const cells = P.map(p=>{
    const v=p[m.key];
    const t = m.type==="div" ? v/hi : (v-lo)/(hi-lo);
    const col = m.type==="div" ? divColor(t) : seqColor(Math.max(0,Math.min(1,t)));
    return `<circle cx="${sx(p.lon).toFixed(1)}" cy="${sy(p.lat).toFixed(1)}" r="4.4"
      fill="${col}" stroke="#fff" stroke-width=".7"
      data-t="${p.ward||''}|${p.burden_mean}|${p.pred}|${p.residual}"></circle>`;}).join("");
  $("#map").innerHTML = `<rect width="${W}" height="${H}" fill="#fbfcfd"/>` + cells;
  $("#mapcap").textContent = m.cap;
  $("#map").querySelectorAll("circle").forEach(c=>{
    c.addEventListener("mousemove",e=>{const d=c.dataset.t.split("|");
      showTip(e,`<b>${d[0]||"—"}</b><br>reported ${d[1]}<br>predicted ${d[2]}<br>residual ${d[3]}`);});
    c.addEventListener("mouseleave",hideTip);});
  const steps=[0,.25,.5,.75,1].map(t=>{
    const col = m.type==="div"?divColor(t*2-1):seqColor(t);
    const v = m.type==="div" ? (lo+(hi-lo)*t).toFixed(2) : (lo+(hi-lo)*t).toFixed(1);
    return `<span><i class="sw" style="background:${col}"></i>${v}</span>`;}).join("");
  $("#maplegend").innerHTML = steps;
}
$("#maptabs").innerHTML = MODES.map((m,i)=>
  `<button class="tab" role="tab" aria-selected="${i===0}" data-i="${i}">${m.label}</button>`).join("");
$("#maptabs").addEventListener("click",e=>{
  const b=e.target.closest(".tab"); if(!b)return;
  mode=+b.dataset.i;
  $("#maptabs").querySelectorAll(".tab").forEach((t,i)=>t.setAttribute("aria-selected", i===mode));
  drawMap();});
drawMap();

// ---------------- legibility
(function(){
  const L=[...DATA.legibility].sort((a,b)=>a.r2_full-b.r2_full);
  const W=470,H=250,PL=122,PR=52,PT=10,PB=28;
  const raw=L.map(d=>d.r2_full), pad=(Math.max(...raw)-Math.min(0,...raw))*0.22;
  const lo=Math.min(0,...raw)-pad, hi=Math.max(...raw)+pad;
  const sxx=v=>PL+(v-lo)/(hi-lo)*(W-PL-PR);
  const bh=(H-PT-PB)/L.length;
  const bars=L.map((d,i)=>{
    const y=PT+i*bh+bh*0.22, h=bh*0.56;
    const xz=sxx(0), xv=sxx(d.r2_full);
    const col=d.r2_full<0?"#b23a48":"#1b4965";
    return `<rect x="${Math.min(xz,xv)}" y="${y}" width="${Math.abs(xv-xz)}" height="${h}"
      fill="${col}" rx="2"><title>${d.issue}: R² = ${d.r2_full}</title></rect>
      <text x="${PL-7}" y="${y+h*0.72}" text-anchor="end" font-size="11" fill="#1a1d21">${d.issue}</text>
      <text x="${xv+(d.r2_full<0?-5:5)}" y="${y+h*0.72}" font-size="10.5" fill="#6b7480"
        text-anchor="${d.r2_full<0?'end':'start'}">${d.r2_full.toFixed(3)}</text>`;}).join("");
  $("#leg").innerHTML = bars +
    `<line x1="${sxx(0)}" y1="${PT}" x2="${sxx(0)}" y2="${H-PB}" stroke="#1a1d21" stroke-width="1"/>
     <text x="${(PL+W-PR)/2}" y="${H-7}" text-anchor="middle" font-size="10.5" fill="#6b7480">block-CV R² (all 45 features)</text>`;
})();

// ---------------- shap
(function(){
  const Sh=DATA.shap, W=470,PL=175,PR=52,PT=8,PB=26,H=300;
  const hi=Math.max(...Sh.map(d=>d.share_pct));
  const bh=(H-PT-PB)/Sh.length;
  const col={"Survey archive":"#1b4965","Demographics":"#c26a3d","Open data":"#6b8f71"};
  $("#shap").innerHTML = Sh.map((d,i)=>{
    const y=PT+i*bh+bh*0.2, h=bh*0.6, w=(d.share_pct/hi)*(W-PL-PR);
    return `<rect x="${PL}" y="${y}" width="${w}" height="${h}" fill="${col[d.block]}" rx="2"/>
      <text x="${PL-7}" y="${y+h*0.75}" text-anchor="end" font-size="10.5" fill="#1a1d21">${d.label}</text>
      <text x="${PL+w+6}" y="${y+h*0.75}" font-size="10" fill="#6b7480">${d.share_pct.toFixed(1)}%</text>`;
  }).join("");
})();

// ---------------- divergence table
(function(){
  const L=DATA.legibility;
  const rank=(key,desc)=>{const s=[...L].sort((a,b)=>desc?b[key]-a[key]:a[key]-b[key]);
    const m={}; s.forEach((d,i)=>m[d.issue]=i+1); return m;}
  const rr=rank("resident_mean",true), re=rank("expert_severity",true),
        ra=rank("authority_agenda_pct",true), rl=rank("r2_full",true);
  const rows=[...L].sort((a,b)=>b.resident_mean-a.resident_mean).map(d=>{
    const spread=Math.max(rr[d.issue],re[d.issue],ra[d.issue])-Math.min(rr[d.issue],re[d.issue],ra[d.issue]);
    return `<tr><td>${d.issue}</td><td>${rr[d.issue]}</td><td>${re[d.issue]}</td>
      <td>${ra[d.issue]}</td><td>${rl[d.issue]}</td>
      <td><span class="bar" style="width:${spread*13+4}px;background:${spread>=3?'#b23a48':'#468faf'}"></span> ${spread}</td></tr>`;}).join("");
  $("#divtable").innerHTML =
    `<thead><tr><th>Burden</th><th>Residents</th><th>Experts</th><th>Official agenda</th>
      <th>Data legibility</th><th>Rank spread</th></tr></thead><tbody>${rows}</tbody>`;
})();
</script></body></html>
"""
HTML = HTML.replace("__PAYLOAD__", json.dumps(PAYLOAD, ensure_ascii=False))
DASH_PATH = DASH / "ttll_decision_dashboard.html"
DASH_PATH.write_text(HTML, encoding="utf-8")
print(f"  -> Dashboard/ttll_decision_dashboard.html ({len(HTML)/1024:.0f} kB, self-contained)")
rec("dashboard_kb", round(len(HTML) / 1024, 1))


  -> Dashboard/ttll_decision_dashboard.html (74 kB, self-contained)


74.5

### Figure 6 — screenshot of the running dashboard

The dashboard is rendered headlessly and captured, so Figure 6 is a picture of software
that ran rather than a mock-up. If Playwright is unavailable the notebook says so and the
screenshot must be taken manually by opening the HTML file; the analysis is unaffected.


In [24]:
FIG6 = FIG / "Figure6_dashboard.png"
try:
    from playwright.sync_api import sync_playwright
    with sync_playwright() as pw:
        br = pw.chromium.launch(args=["--no-sandbox"])
        pg = br.new_page(viewport={"width": 1400, "height": 1350}, device_scale_factor=2)
        pg.goto(DASH_PATH.resolve().as_uri())
        pg.wait_for_timeout(1400)
        pg.screenshot(path=str(FIG6), full_page=True)
        br.close()
    print(f"  -> Figures/Figure6_dashboard.png ({FIG6.stat().st_size/1024:.0f} kB)")
    rec("figure6_captured", True)
except Exception as exc:
    print(f"  ! headless capture unavailable ({type(exc).__name__}). "
          f"Open {DASH_PATH.name} in a browser and screenshot it for Figure 6.")
    rec("figure6_captured", False)


  ! headless capture unavailable (Error). Open ttll_decision_dashboard.html in a browser and screenshot it for Figure 6.


## Step 13 — Write the results ledger

Every scalar reported anywhere in the manuscript is written here, so that a reviewer (or a
future co-author) can trace any number in the text back to the line of code that made it.


In [25]:
def _clean(o):
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    return o


with open(RES / "paper4_results.json", "w", encoding="utf-8") as fh:
    json.dump(_clean(RESULTS), fh, ensure_ascii=False, indent=2)
print(f"  -> Results/paper4_results.json  ({len(RESULTS)} recorded values)")

idx = []
for name, meta in RESULTS.get("_tables", {}).items():
    idx.append({"table": name, "file": f"Tables/{name}.csv",
                "rows": meta["rows"], "caption": meta["caption"]})
IDX = pd.DataFrame(idx).sort_values("table")
IDX.to_csv(TAB / "_INDEX.csv", index=False, encoding="utf-8-sig")
print(f"  -> Tables/_INDEX.csv  ({len(IDX)} tables)")

FIGS = sorted(p.name for p in FIG.glob("*.png"))
print(f"\nfigures written: {len(FIGS)}")
for f in FIGS:
    print("   ", f)
print(f"\ntables written : {len(IDX)}")
print("\nSTEP-BY-STEP HEADLINE SUMMARY")
print("=" * 78)
for k in ["alpha_burden", "moran_flood", "inst_share_pct",
          "dt_progress_expert_mean", "dt_progress_auth_mean", "dt_progress_p",
          "eval_tools_expert_agree_pct", "eval_tools_auth_agree_pct", "eval_tools_p",
          "best_block_model", "best_block_r2", "best_block_mae",
          "moran_residual", "moran_structure_removed_pct",
          "transfer_best_rho", "transfer_best_rho_p",
          "legibility_best", "legibility_worst",
          "rho::residents vs data legibility"]:
    if k in RESULTS:
        print(f"  {k:42s} {RESULTS[k]}")
print("=" * 78)


  -> Results/paper4_results.json  (192 recorded values)
  -> Tables/_INDEX.csv  (22 tables)

figures written: 7
    Figure1_pipeline.png
    Figure2_barrier_diagnosis.png
    Figure3_model_ladder.png
    Figure4_shap.png
    Figure5_spatial_accountability.png
    Figure6_dashboard.png
    Figure7_institutional_gap.png

tables written : 22

STEP-BY-STEP HEADLINE SUMMARY
  alpha_burden                               0.846
  moran_flood                                0.1834
  inst_share_pct                             77.2
  dt_progress_expert_mean                    2.93
  dt_progress_auth_mean                      3.43
  dt_progress_p                              0.0261
  eval_tools_expert_agree_pct                14.6
  eval_tools_auth_agree_pct                  47.8
  eval_tools_p                               0.0033
  best_block_model                           M3  demographics + survey archive (OLS)
  best_block_r2                              0.2924
  best_block_mae                  